# Video Face Swap - 1 người + **nhuộm tóc** (inswapper + InsightFace + MediaPipe)

**Logic:** 1 video mẫu + 1 ảnh khuôn mặt → video mới giữ nguyên chuyển động/nền gốc, thay
**khuôn mặt** và **đổi màu tóc** (giữ nguyên kiểu tóc của người trong video).

## Đúng, việc này dễ hơn hẳn — và không chỉ dễ hơn, mà *đúng hơn*

Bản `video_face_swap_1nguoi_toc.ipynb` thay cả **kiểu** tóc: phải cắt mảng tóc từ ảnh nguồn rồi
dán lên đầu người trong video. Đổi **màu** thì không cần dán gì cả: tóc đã có sẵn ở đúng chỗ,
đúng hình, đúng chuyển động — chỉ cần **đổi màu của những pixel đó**.

Bốn thứ khó nhất của bản kia biến mất hoàn toàn, không phải "làm đơn giản đi" mà là **không còn
tồn tại**:

| Bản thay kiểu tóc phải làm | Bản đổi màu |
|---|---|
| Warp mảng tóc 2D theo 5 điểm mốc (Umeyama + EMA ma trận) | **không cần** — tóc vốn đã ở đúng chỗ |
| Mờ dần tóc khi quay đầu (`HAIR_MAX_TURN`) vì tóc 2D dán lên đầu nghiêng sẽ vỡ | **không cần** — quay đầu, cúi, quay lưng đều đổi màu đúng |
| Inpaint phần tóc cũ thò ra khi tóc nguồn ngắn hơn | **không cần** — không có "tóc cũ thừa" |
| Hộp `HAIR_ROI` quanh đầu, tóc ngoài hộp thì không xử lý được | **không cần** — chạy trên cả khung, tóc dài ngang lưng vẫn ăn |

**Hệ quả đáng giá nhất:** phần tóc ở đây **không phụ thuộc vào việc detect được mặt**. Frame nào
người quay lưng, cúi đầu, mặt bị che — face swap bỏ qua, nhưng tóc vẫn đổi màu bình thường.
Ở bản thay kiểu tóc thì mất mặt là mất luôn tóc.

## Đường đi của một khung hình

```
frame gốc
  → InsightFace detect  →  inswapper swap mặt  →  GFPGAN làm nét   (bỏ qua nếu không thấy mặt)
  → NHUỘM TÓC:  segment tóc → đổi màu trong không gian LAB → blend  (luôn luôn chạy)
  → ffmpeg
```

## Đổi màu bằng cách nào

Segment vùng tóc bằng MediaPipe (`selfie_multiclass_256x256`, có sẵn lớp `hair`), rồi làm việc
trong **không gian màu LAB** — nơi độ sáng `L` tách rời khỏi màu sắc `a`/`b`.

Đây là chỗ quyết định kết quả nhìn thật hay nhìn giả:

- **Giữ nguyên toàn bộ biến thiên của `L`** (sợi tóc, bóng đổ, ánh sáng phản chiếu), chỉ **dịch**
  cả phân bố lên/xuống. Nhuộm mà xoá mất biến thiên độ sáng thì ra một mảng màu phẳng lì dán
  lên đầu — lỗi kinh điển của cách nhuộm bằng cách tô đè màu.
- **Thay `a`/`b` bằng màu đích**, giữ lại một phần biến thiên gốc (`HAIR_KEEP_TONE`) để không bị
  "màu sơn".
- **Chừa lại vùng phản chiếu sáng** (`HAIR_KEEP_HIGHLIGHT`): ánh sáng phản chiếu trên tóc thật
  gần như không màu. Nhuộm cả vào đó thì tóc mất độ bóng, trông như tóc giả.

> **Giới hạn vật lý, không phải giới hạn của code:** tóc đen trong video nhuộm sang vàng bạch
> kim sẽ không đẹp bằng nhuộm sang nâu. Pixel gần đen thì gần như không còn chi tiết để giữ —
> ngoài đời cũng phải tẩy tóc trước mới nhuộm sáng được, ở đây cũng vậy. `HAIR_LIGHTNESS` là
> nút chỉnh chỗ này.

⚠️ Công nghệ face-swap có thể bị dùng sai mục đích. Chỉ dùng với ảnh/video của chính bạn hoặc
người đã đồng ý.

**Trước khi chạy:** Runtime > Change runtime type > chọn GPU (T4).

## 0. Cấu hình

| | `True` | `False` |
|---|---|---|
| `USE_GFPGAN` | cài `gfpgan/facexlib/basicsr` (phải vá source), tải `GFPGANv1.4.pth` (~340MB), làm nét mặt mỗi frame | không cài, không tải |
| `USE_HAIR` | cài `mediapipe`, tải `selfie_multiclass_256x256.tflite` (~16MB), nhuộm tóc mỗi frame | giữ nguyên màu tóc trong video |

### Chọn màu: `HAIR_COLOR` nhận ba kiểu giá trị

1. **Tên trong bảng màu** — `'nâu hạt dẻ'`, `'vàng đồng'`, `'đỏ rượu'`... (bảng ở cell dưới)
2. **Mã hex bất kỳ** — `'#8B5A2B'`
3. **`'ảnh'`** — lấy **màu tóc trung bình của người trong ảnh nguồn** bạn upload. Hợp lý nhất
   khi bạn muốn video trông giống hệt người trong ảnh: cùng mặt, cùng màu tóc.

> ⚠️ **Mặc định là `'nâu hạt dẻ'`, tức là một màu cố định — KHÔNG liên quan gì tới ảnh nguồn.**
> Nếu bạn muốn tóc trong video giống tóc người trong ảnh thì phải tự đổi thành `'ảnh'`.

### Sáu nút quyết định kết quả

- **`HAIR_LIGHTNESS`** (0→1): kéo độ sáng tóc về màu đích. Đây là nút **quan trọng nhất**, và
  là chỗ hay gây bất ngờ nhất: nó quyết định tóc có **tới được** màu bạn chọn hay không. Tóc đen
  (L≈40) nhuộm sang trắng (L≈230) mà để `0.55` thì tóc chỉ đi nửa đường, dừng ở L≈143 — ra
  **xám**, không ra trắng. Chênh lệch càng lớn càng phải để gần `1.0`. Cell 6.1 tự đo và cảnh
  báo trước chuyện này.
  `0` = giữ nguyên độ sáng gốc, chỉ đổi sắc màu (hợp với đen→nâu đỏ, nâu→hạt dẻ).
  `1` = bám hoàn toàn độ sáng màu đích (cần cho đen→vàng/bạch kim, nhưng dễ trông "dán vào" nếu
  video thiếu sáng, vì lúc đó tóc sáng hơn cả khung cảnh).
- **`HAIR_KEEP_TONE`** (0→1): giữ bao nhiêu biến thiên màu gốc. `0` = màu đích tuyệt đối phẳng
  (sạch nhưng dễ giả), `0.3` = còn chút sắc độ tự nhiên.
- **`HAIR_KEEP_HIGHLIGHT`** (0→1): chừa vùng phản chiếu sáng không nhuộm. Giảm nếu thấy tóc bị
  bạc trắng ở chỗ bắt sáng; tăng nếu tóc trông bết, mất độ bóng.
- **`HAIR_WISPS`**: nhuộm cả những **sợi tóc mai** mảnh rủ trước mặt. Chúng rộng 4-8 pixel, tức
  chưa tới một ô của mask 256×256, nên MediaPipe không thấy — mà lại còn bị `HAIR_PROTECT_FACE`
  trừ mất vì nằm trên da mặt. Tầng này bắt riêng chúng; mắt/mũi/miệng/lông mày đã chừa sẵn.
- **`HAIR_EDGE_SMART`**: xử lý **mép tóc**. Mask 256×256 quá thô ở mép (trên 1080p mỗi ô mask
  thành hơn 4 pixel ảnh), nên nếu chỉ dựa vào mask thì chỉ có hai kết cục, đều tệ: **quầng sáng**
  quanh đầu, hoặc **viền tối** còn nguyên màu tóc cũ. Bật cái này thì độ phủ ở mép được ước
  lượng từ chính độ sáng của ảnh (pixel mép là màu pha tóc/nền), nên tránh được cả hai. Để `True`.
- **`HAIR_CONF`**: dốc mềm của ngưỡng mask. Nới rộng xuống nếu tóc bị nhuộm hụt ở rìa, kéo lên
  nếu màu lem sang nền/vai.

**Không cần hiểu hết ngay.** Chạy tới mục 6.1 (thử 1 frame) sẽ thấy ngay kết quả — sửa rồi chạy
lại hai cell, vài giây một vòng, không phải xử lý cả video.

### Về tốc độ

Nhuộm tóc thêm **một lần segment MediaPipe ở 256×256 (chi phí cố định, không phụ thuộc độ phân
giải video)** + phần đổi màu **chỉ chạy trong hộp bao quanh vùng tóc**, không phải cả khung.
Rẻ hơn GFPGAN nhiều. Cell cuối mục 6 in thời gian từng bước để bạn biết chính xác cái nào tốn.

In [ ]:
# ===================== CÔNG TẮC CHÍNH =====================
USE_GFPGAN = True    # True  = swap xong làm nét mặt (đẹp hơn, chậm hơn)
USE_HAIR   = True    # True  = đổi màu tóc của người trong video
# ==========================================================


# ===================== MÀU TÓC =====================
# Nhận 3 kiểu: tên trong HAIR_PALETTE | mã hex '#RRGGBB' | 'ảnh' (lấy màu tóc từ ẢNH NGUỒN)
#
# LƯU Ý: đặt TÊN MÀU ở đây thì tóc ra ĐÚNG MÀU ĐÓ, KHÔNG liên quan gì tới ảnh nguồn.
# 'ảnh' = lấy màu tóc của chính người trong ảnh bạn upload (mặt và tóc cùng một người).
HAIR_COLOR = 'ảnh'

HAIR_PALETTE = {
    'đen':            '#1B1917',
    'nâu đen':        '#2E211B',
    'nâu socola':     '#43291B',
    'nâu hạt dẻ':     '#5A3220',
    'nâu tây':        '#8B5A2B',
    'nâu khói':       '#6B5A4E',
    'vàng đồng':      '#A9762F',
    'vàng mật ong':   '#C08D4A',
    'vàng bạch kim':  '#D8C8A2',
    'bạch kim':       '#CFC7BB',
    'trắng':          '#E5E3DE',
    'bạc':            '#9A9A9A',
    'đỏ rượu':        '#5E1F22',
    'đỏ cam':         '#8E3B1E',
    'hồng khói':      '#9C6A6E',
    'tím khói':       '#4A3350',
    'xanh rêu':       '#3A4232',
    'xanh khói':      '#37474F',
}
# ===================================================


# ============ TINH CHỈNH (chỉ có tác dụng khi USE_HAIR = True) ============
# Chỉnh xong chạy lại cell 6.1 (thử 1 frame) để xem ngay, không cần chạy cả video.

HAIR_LIGHTNESS      = 1.0    # 0-1: kéo ĐỘ SÁNG về màu đích. NÚT QUAN TRỌNG NHẤT.
                             # 0   = giữ nguyên độ sáng gốc, chỉ đổi sắc màu
                             # 1   = bám hoàn toàn độ sáng màu đích  <-- đang để mức này
                             #
                             # Đây là chỗ hay gây bất ngờ nhất: nhuộm tóc ĐEN (L~40) sang
                             # TRẮNG (L~230) mà để 0.55 thì tóc chỉ đi được nửa đường, dừng ở
                             # L~143 -> ra XÁM chứ không ra trắng. Chênh lệch càng lớn càng
                             # phải để gần 1.0.
                             # Hạ xuống 0.6-0.8 nếu thấy tóc sáng quá so với ánh sáng cảnh quay
                             # (chỉ hợp khi màu đích không sáng hơn tóc gốc quá nhiều).

HAIR_KEEP_TONE      = 0.30   # 0-1: giữ bao nhiêu biến thiên màu gốc. 0 = màu đích phẳng lì.

HAIR_KEEP_HIGHLIGHT = 0.35   # 0-1: chừa bao nhiêu phần MÀU ở vùng tóc bắt sáng (ánh phản
                             # chiếu trên tóc thật gần như không màu, nhuộm hết vào đó thì tóc
                             # mất bóng, trông như tóc giả).
                             # Nhưng để CAO quá thì chỗ bóng nhất - đỉnh đầu, chỗ rẽ ngôi -
                             # nhìn như còn sót màu tóc cũ. 0.3-0.4 là chỗ cân bằng.
                             # Tóc bết, mất bóng -> tăng. Còn thấy mảng màu cũ -> giảm về 0.2.

HAIR_CONTRAST       = 1.30   # nhân biến thiên độ sáng. >1 = rõ sợi tóc hơn. Cần khi nhuộm tóc
                             # ĐEN sang màu SÁNG: pixel gần đen vốn rất ít chi tiết, kéo sáng
                             # lên mà không nhân tương phản thì ra mảng bệt. 1.2-1.4 là đủ.
                             # Về 1.0 nếu tóc gốc và màu đích sáng gần bằng nhau.

HAIR_SATURATION     = 1.00   # nhân độ rực của màu đích. <1 = nhạt đi, >1 = rực hơn.

HAIR_STRENGTH       = 1.00   # độ đậm tổng thể. <1 = pha với màu tóc gốc (0.6-0.8 cho tự nhiên).

HAIR_CONF           = (0.35, 0.65)   # dốc mềm của mask: dưới 0.35 không nhuộm, trên 0.65 nhuộm
                             # hết, ở giữa chuyển dần. Nhuộm hụt rìa tóc -> hạ xuống (0.25,0.5).
                             # Màu lem sang nền/vai -> nâng lên (0.5, 0.8).

HAIR_FEATHER        = 2.0    # làm mềm mép mask (đơn vị: pixel ở không gian 256x256, nên nó tự
                             # co giãn theo độ phân giải video).

HAIR_WISPS          = 0.7    # 0-1: nhuộm cả những SỢI TÓC MAI mảnh rủ trước mặt.
                             # Mask 256x256 không thấy nổi sợi rộng 4-8 pixel (chưa tới một ô
                             # mask), lại còn bị HAIR_PROTECT_FACE trừ mất vì chúng nằm trên
                             # da mặt -> cả mái tóc đổi màu mà mấy sợi trước mặt vẫn đen.
                             # Tầng này bắt riêng chúng bằng độ tương phản sáng/tối.
                             # Mắt, mũi, miệng, lông mày đã được loại trừ sẵn.
                             # 0 = tắt. Nhuộm nhầm vào vệt tối trên mặt/cổ -> giảm.

HAIR_EDGE_SMART     = True   # Xử lý MÉP TÓC bằng chính độ sáng của ảnh, thay vì chỉ dựa vào
                             # mask 256x256 (vốn quá thô ở mép: phóng lên 1080p thì mỗi ô mask
                             # thành hơn 4 pixel ảnh).
                             #
                             # Tắt đi thì quay lại đúng hai lựa chọn tồi: mask rộng -> QUẦNG
                             # SÁNG viền tóc, mask hẹp -> VIỀN TỐI còn nguyên màu tóc cũ.
                             # Bật thì mép tóc được ước lượng theo tỉ lệ pha tóc/nền của từng
                             # pixel, nên không dính cái nào trong hai cái đó.
                             #
                             # Chỉ bất lực khi tóc và nền sáng xấp xỉ nhau (tóc đen trên nền
                             # tối) - lúc đó nó tự quay về dùng mask như cũ.

HAIR_EDGE_CHOKE     = 0      # CO mask vào trong ngần này pixel trước khi làm mềm (đơn vị như
                             # HAIR_FEATHER). Là cách chữa quầng sáng CŨ, thô hơn: nó ăn luôn
                             # cả mép tóc thật nên hay để lại viền tối - trên video 1080p thì
                             # mỗi đơn vị ở đây là hơn 4 pixel ảnh, khá dày.
                             # Để 0 khi HAIR_EDGE_SMART = True. Chỉ dùng khi bạn tắt SMART.

HAIR_MASK_SMOOTH    = 0.5    # 0-1: làm mượt mask theo thời gian (EMA) cho đỡ nhấp nháy.
                             # 0 = tắt, 0.8 = rất mượt nhưng trễ khi chuyển động nhanh.

HAIR_PROTECT_FACE   = True   # trừ vùng da mặt ra khỏi mask, tránh nhuộm lem lên trán/lông mày.
# ==========================================================================

print('USE_GFPGAN =', USE_GFPGAN, '| USE_HAIR =', USE_HAIR)
if USE_HAIR:
    if HAIR_COLOR == 'ảnh':
        print("Màu tóc: lấy trung bình từ ẢNH NGUỒN (tính ở mục 5b sau khi upload ảnh).")
    elif HAIR_COLOR in HAIR_PALETTE:
        print(f"Màu tóc: {HAIR_COLOR!r} = {HAIR_PALETTE[HAIR_COLOR]}")
    elif isinstance(HAIR_COLOR, str) and HAIR_COLOR.startswith('#'):
        print(f'Màu tóc: mã hex {HAIR_COLOR}')
    else:
        # Báo ngay tại đây thay vì để tới mục 5b mới nổ, lúc đó đã cài/tải xong hết rồi.
        raise ValueError(
            f'HAIR_COLOR = {HAIR_COLOR!r} không hợp lệ.\n'
            f"Dùng 'ảnh', mã hex '#RRGGBB', hoặc một trong: {', '.join(HAIR_PALETTE)}"
        )
    print(f'  độ sáng {HAIR_LIGHTNESS} | giữ sắc gốc {HAIR_KEEP_TONE} | '
          f'chừa vùng bóng {HAIR_KEEP_HIGHLIGHT} | đậm {HAIR_STRENGTH}')
    if HAIR_COLOR != 'ảnh':
        print("  (muốn lấy đúng màu tóc của người trong ảnh nguồn: đặt HAIR_COLOR = 'ảnh')")
else:
    print('-> giữ nguyên màu tóc trong video.')

## 1. Cài đặt thư viện

In [ ]:
# Log phiên bản Python Colab đang cấp (để biết đang chạy bản nào)
import sys, platform
print('Python version:', sys.version)
print('Platform:', platform.platform())

In [ ]:
# Ghim setuptools < 82: bản setuptools mới (>=82) đã bỏ hẳn module 'distutils',
# trong khi basicsr (dependency của GFPGAN) và torch trên Colab vẫn cần distutils/setuptools cũ.
!pip install -q "setuptools==79.0.1" wheel
!pip install -q cython numpy

# --no-build-isolation: để insightface dùng đúng cython/numpy vừa cài ở trên,
# thay vì pip tự tạo môi trường tạm cô lập (không thấy cython) rồi build lỗi 'egg_info'.
!pip install -q --no-build-isolation insightface==0.7.3

# KHÔNG cài opencv-python-headless: Colab đã có sẵn opencv-python, mà hai package này
# dùng chung namespace `cv2` -> cài đè lên nhau hay để lại .so lẫn lộn gây lỗi khó hiểu.
# (insightface cũng khai báo opencv-python là dependency nên chắc chắn có cv2.)

# gfpgan/facexlib chỉ cần khi làm nét. Tắt thì bỏ hẳn -> nhanh hơn và bớt một nguồn lỗi.
EXTRA_PKGS = 'gfpgan facexlib' if USE_GFPGAN else ''
print('Cài thêm:', EXTRA_PKGS or '(không có, USE_GFPGAN = False)')
!pip install -q onnxruntime-gpu {EXTRA_PKGS}

# libportaudio2: mediapipe import `sounddevice`, thư viện này ném OSError ngay lúc import nếu
# thiếu PortAudio trên máy -> `import mediapipe` chết dù package cài thành công. Ảnh Colab có
# lúc có lúc không, cài luôn cho chắc (vài trăm KB).
!apt-get -qq install -y ffmpeg libportaudio2 > /dev/null
print('Xong.')

### Cài `mediapipe` mà không đụng vào `cv2` (chỉ chạy khi `USE_HAIR = True`)

`pip install mediapipe` kéo theo **`opencv-contrib-python`**. Package đó chiếm đúng namespace
`cv2` mà `opencv-python` (Colab cài sẵn, insightface đang dùng) đang chiếm — cài đè lên nhau là
đúng cái lỗi `cv2` lẫn lộn mà notebook gốc đã ghi chú tránh ở cell trên. Nó cũng hay ghim lại
`numpy`/`protobuf`, làm hỏng ngược `onnxruntime` vừa cài.

Nên cell dưới cài `--no-deps` rồi **tự cài đúng những dependency thật sự cần** cho Image
Segmenter (`absl-py`, `attrs`, `flatbuffers`, `protobuf`, `jax` không cần). `cv2` thì dùng lại
bản Colab đã có.

Nếu bản `--no-deps` không import được, cell **tự fallback sang `pip install mediapipe` đầy đủ**
và in cảnh báo — vẫn chạy được, chỉ là môi trường bẩn hơn. Nếu cả hai đều hỏng thì
`HAIR_AVAILABLE = False` và pipeline chạy tiếp **không thay tóc** (giống cách `gfpgan` fallback),
chứ không làm chết cả notebook.

In [ ]:
import subprocess, sys, importlib

HAIR_AVAILABLE = False


def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-2500:])
    print(r.stderr[-2500:])
    return r.returncode


def try_import_mediapipe():
    """Import thử cả `mediapipe` lẫn đúng submodule Image Segmenter sẽ dùng.

    Import mỗi `mediapipe` là chưa đủ: `mediapipe.tasks.python.vision` mới là chỗ ném lỗi khi
    thiếu dependency, mà nó chỉ được nạp khi gọi tới -> phải thử ngay tại đây.
    """
    importlib.invalidate_caches()
    try:
        import mediapipe as mp
        from mediapipe.tasks.python import vision as _vision   # noqa: F401
        print('mediapipe OK, version:', getattr(mp, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception chứ không chỉ ImportError: thiếu PortAudio ném OSError,
        # lệch protobuf ném TypeError/AttributeError -> chỉ bắt ImportError thì cell crash.
        print(f'Chưa import được mediapipe: {type(e).__name__}: {e}')
        return False


if not USE_HAIR:
    print('USE_HAIR = False -> bỏ qua mediapipe (chỉ cần cho phần tóc).')
else:
    HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print('Cài mediapipe --no-deps (giữ nguyên cv2/numpy của Colab)...')
        pip('install', '-q', '--no-deps', 'mediapipe')
        # Dependency tối thiểu cho Tasks API. Không đụng numpy/opencv/protobuf-version.
        pip('install', '-q', 'absl-py', 'attrs', 'flatbuffers', 'sentencepiece', 'sounddevice')
        HAIR_AVAILABLE = try_import_mediapipe()

    if not HAIR_AVAILABLE:
        print()
        print('Bản --no-deps không chạy -> fallback: cài mediapipe đầy đủ.')
        print('CẢNH BÁO: bước này có thể cài đè opencv-contrib-python lên cv2 hiện có.')
        pip('install', 'mediapipe')
        HAIR_AVAILABLE = try_import_mediapipe()

        # Cài đè cv2 xong phải kiểm tra lại chính cv2 + onnxruntime, vì đó là thứ dễ vỡ nhất.
        for mod in ('cv2', 'onnxruntime'):
            try:
                m = importlib.import_module(mod)
                print(f'  {mod} vẫn OK, version:', getattr(m, '__version__', '?'))
            except Exception as e:
                print(f'  !! {mod} HỎNG sau khi cài mediapipe: {type(e).__name__}: {e}')
                print('     Runtime > Restart session rồi chạy lại từ đầu notebook.')

    if not HAIR_AVAILABLE:
        print()
        print('mediapipe không cài được -> sẽ chạy tiếp mà KHÔNG thay tóc.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

### Fix riêng cho `basicsr` (chỉ chạy khi `USE_GFPGAN = True`)

`basicsr` có bug trong `setup.py`: dùng `exec(...)` rồi đọc `locals()['__version__']` để lấy
version. Ở Python 3.13, `exec()` trong 1 hàm không còn ghi ngược lại `locals()` đáng tin cậy
(thay đổi theo PEP 667) → lỗi `KeyError: '__version__'`. Cell dưới tải source về, patch đúng
chỗ này (`locals()` → `globals()`), rồi cài từ bản đã sửa.

`basicsr` chỉ là dependency của GFPGAN, nên `USE_GFPGAN = False` thì cell này không làm gì cả.

In [ ]:
import subprocess, sys, os, tarfile, urllib.request, json


def install_basicsr_patched():
    """Tải basicsr từ PyPI, vá bug PEP 667 trong setup.py, rồi cài từ source đã sửa."""
    os.makedirs('/tmp/basicsr_src', exist_ok=True)

    # Tải trực tiếp từ PyPI bằng urllib (KHÔNG dùng `pip download`, vì pip cũng phải chạy
    # setup.py egg_info để lấy metadata -> dính đúng bug KeyError trước khi kịp patch).
    with urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json') as resp:
        pkg_info = json.load(resp)

    sdist_url = None
    for url_info in pkg_info['urls']:
        if url_info['packagetype'] == 'sdist':
            sdist_url = url_info['url']
            break
    assert sdist_url is not None, 'Không tìm thấy sdist của basicsr trên PyPI.'

    tar_name = sdist_url.split('/')[-1]
    tar_path = f'/tmp/basicsr_src/{tar_name}'
    urllib.request.urlretrieve(sdist_url, tar_path)
    print(f'Đã tải: {tar_name}')

    extract_dir = '/tmp/basicsr_build'
    os.makedirs(extract_dir, exist_ok=True)
    with tarfile.open(tar_path) as tar:
        # Lấy tên thư mục gốc từ chính nội dung tar thay vì suy ra bằng tar_name.replace(...),
        # vì sdist trên PyPI không bắt buộc phải là .tar.gz.
        root_names = {m.name.split('/')[0] for m in tar.getmembers() if m.name.strip('./')}
        assert len(root_names) == 1, f'sdist có cấu trúc thư mục lạ: {root_names}'
        root_name = root_names.pop()
        # filter='data': Python 3.12+ deprecate extractall không có filter, 3.14 đổi default.
        tar.extractall(extract_dir, filter='data')

    pkg_dir = os.path.join(extract_dir, root_name)
    setup_py_path = os.path.join(pkg_dir, 'setup.py')
    assert os.path.exists(setup_py_path), f'Không thấy setup.py trong {pkg_dir}'

    with open(setup_py_path, 'r') as f:
        content = f.read()

    # Fix bug Python 3.13: exec() trong hàm không ghi ngược locals() đáng tin cậy
    content = content.replace(
        "exec(compile(f.read(), version_file, 'exec'))",
        "exec(compile(f.read(), version_file, 'exec'), globals())"
    )
    content = content.replace(
        "return locals()['__version__']",
        "return globals()['__version__']"
    )

    with open(setup_py_path, 'w') as f:
        f.write(content)

    print('Đã patch setup.py xong, tiến hành cài basicsr từ source đã sửa...')
    # sys.executable -m pip: đảm bảo cài vào đúng interpreter đang chạy notebook,
    # thay vì lệnh `pip` bất kỳ đứng đầu PATH.
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', pkg_dir],
        capture_output=True, text=True
    )
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError('Cài basicsr thất bại, xem log lỗi ở trên để biết nguyên nhân cụ thể.')
    print('Cài basicsr thành công.')


if USE_GFPGAN:
    install_basicsr_patched()
else:
    print('USE_GFPGAN = False -> bỏ qua basicsr (chỉ là dependency của GFPGAN).')

## 2. Tải model

Model `inswapper_128.onnx` không được host chính thức trên GitHub release nữa do vấn đề chính
sách, nên bạn cần tự tải và upload lên Google Drive của mình, hoặc dùng link mirror cộng đồng
(huggingface). Cell dưới thử tải từ 1 mirror phổ biến trên Hugging Face — nếu lỗi, bạn tải thủ
công rồi upload vào `/content/models/`.

| Model | Khi nào tải | Dung lượng |
|---|---|---|
| `inswapper_128.onnx` | luôn luôn | ~530 MB |
| `GFPGANv1.4.pth` | `USE_GFPGAN = True` | ~340 MB |
| `selfie_multiclass_256x256.tflite` | `USE_HAIR = True` | ~16 MB |

Model segment tải **thẳng từ `storage.googleapis.com/mediapipe-models`** — kho chính thức của
Google cho MediaPipe Tasks, không phải mirror cộng đồng. Nó phân 6 lớp:
`0 background, 1 hair, 2 body-skin, 3 face-skin, 4 clothes, 5 others`. Notebook này dùng
**lớp 1 (hair)** để biết nhuộm chỗ nào và **lớp 3 (face-skin)** để trừ ra, khỏi nhuộm lem lên da.

In [ ]:
import os, subprocess
os.makedirs('/content/models', exist_ok=True)

INSWAPPER_PATH = '/content/models/inswapper_128.onnx'
GFPGAN_PATH    = '/content/models/GFPGANv1.4.pth'
SEGMENTER_PATH = '/content/models/selfie_multiclass_256x256.tflite'

INSWAPPER_URL = 'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx'
GFPGAN_URL    = 'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'
# Kho model chính thức của MediaPipe Tasks (Google), không phải mirror cộng đồng.
SEGMENTER_URL = ('https://storage.googleapis.com/mediapipe-models/image_segmenter/'
                 'selfie_multiclass_256x256/float32/latest/selfie_multiclass_256x256.tflite')


# Dùng subprocess thay vì `!wget`: lệnh `!` nằm trong khối `if` phụ thuộc vào chi tiết
# transform của IPython, còn subprocess thì chạy giống nhau ở mọi môi trường và
# trả về returncode để kiểm tra.
def download(url, path, min_mb, hint):
    """Tải file rồi KIỂM TRA DUNG LƯỢNG.

    wget vẫn coi là "thành công" khi server trả 404 / trang HTML -> phải tự kiểm tra.
    Nếu không, mãi tới cell load model mới nổ với lỗi onnx/torch/tflite rất khó đoán nguyên nhân.
    """
    print(f'Đang tải {os.path.basename(path)} ...')
    subprocess.run(['wget', '-q', '-O', path, url])
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    assert size_mb >= min_mb, (
        f'Tải {os.path.basename(path)} thất bại (chỉ {size_mb:.1f} MB, cần >= {min_mb} MB). {hint}'
    )
    print(f'OK  {os.path.basename(path)}: {size_mb:.1f} MB')


download(INSWAPPER_URL, INSWAPPER_PATH, 200,
         'Link mirror có thể đã chết -> tự tải rồi upload vào /content/models/.')

if USE_GFPGAN:
    download(GFPGAN_URL, GFPGAN_PATH, 300,
             'Kiểm tra lại link GitHub release của GFPGAN.')
else:
    print('Bỏ qua GFPGANv1.4.pth (~340MB) vì USE_GFPGAN = False.')

if USE_HAIR:
    download(SEGMENTER_URL, SEGMENTER_PATH, 5,
             'Kiểm tra lại đường dẫn trong kho mediapipe-models của Google.')
else:
    print('Bỏ qua selfie_multiclass_256x256.tflite vì USE_HAIR = False.')

!ls -lh /content/models/

## 3. Upload ảnh khuôn mặt + video mẫu

Chỉ còn **1 ảnh + 1 video**. Không cần quy ước ai là A ai là B, không cần lo upload nhầm thứ tự.

In [ ]:
from google.colab import files

def pick_one(uploaded, what):
    names = list(uploaded.keys())
    assert len(names) > 0, f'Chưa upload {what} (bấm Cancel?). Chạy lại cell này.'
    if len(names) > 1:
        print(f'  (đã upload {len(names)} file, dùng file đầu tiên: {names[0]})')
    return names[0]

print('>> Upload ảnh khuôn mặt (rõ mặt, chính diện càng tốt):')
source_face_path = pick_one(files.upload(), 'ảnh mặt')

print()
print('>> Upload video mẫu:')
source_video_path = pick_one(files.upload(), 'video mẫu')

print()
print(f'Ảnh mặt : {source_face_path}')
print(f'Video   : {source_video_path}')

## 4. Khởi tạo model face analysis + face swapper

### Fix riêng cho `onnxruntime-gpu` (có thể chưa có wheel cho Python bản mới trên Colab)

Cell dưới kiểm tra xem `onnxruntime` đã import được chưa. Nếu chưa, sẽ thử cài lại
`onnxruntime-gpu` với log đầy đủ; nếu vẫn thất bại (do chưa có wheel tương thích Python 3.13),
sẽ **fallback sang `onnxruntime` bản CPU** để pipeline vẫn chạy được (chỉ chậm hơn).

In [ ]:
import subprocess, sys, importlib

def try_import_onnxruntime():
    # Bắt Exception chứ không chỉ ImportError: onnxruntime-gpu thiếu libcudnn/libcublas
    # thường ném OSError/RuntimeError -> nếu chỉ bắt ImportError thì cell crash thay vì fallback.
    importlib.invalidate_caches()
    try:
        import onnxruntime
        print('onnxruntime OK, version:', onnxruntime.__version__)
        print('Available providers:', onnxruntime.get_available_providers())
        return True
    except Exception as e:
        print(f'Chưa import được onnxruntime: {type(e).__name__}: {e}')
        return False

def pip(*args):
    r = subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)
    print(r.stdout[-3000:])
    print(r.stderr[-3000:])
    return r.returncode

if not try_import_onnxruntime():
    # onnxruntime và onnxruntime-gpu cùng chiếm package `onnxruntime`. Cài cái này đè cái kia
    # là trạng thái hỏng đã biết (mất CUDAExecutionProvider, import lỗi loạn)
    # -> luôn gỡ sạch cả hai trước khi cài lại.
    print('Gỡ sạch onnxruntime cũ rồi cài lại onnxruntime-gpu...')
    pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
    pip('install', 'onnxruntime-gpu==1.20.0')

    if not try_import_onnxruntime():
        print('onnxruntime-gpu không cài được (khả năng chưa có wheel cho bản Python này).')
        print('Fallback sang onnxruntime bản CPU...')
        pip('uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu')
        pip('install', 'onnxruntime')
        assert try_import_onnxruntime(), 'Vẫn không cài được onnxruntime, xem log lỗi ở trên.'

In [ ]:
import cv2
import insightface
from insightface.app import FaceAnalysis

import onnxruntime
available_providers = onnxruntime.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in available_providers
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']
# ctx_id phải khớp với providers: 0 = GPU 0, -1 = CPU. Để nguyên 0 khi chỉ có CPU provider là mâu thuẫn.
ctx_id = 0 if USE_CUDA else -1
print('Dùng providers:', providers, '| ctx_id =', ctx_id)

app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=ctx_id, det_size=(640, 640))

swapper = insightface.model_zoo.get_model(INSWAPPER_PATH, download=False, providers=providers)

# cv2.imread trả None khi file hỏng / định dạng không hỗ trợ (heic, webp lạ...).
# Phải chặn ngay, không thì app.get(None) ném lỗi cv2 không nói gì về nguyên nhân thật.
source_img = cv2.imread(source_face_path)
assert source_img is not None, (
    f'Không đọc được ảnh {source_face_path}. Lưu lại thành .jpg/.png rồi upload lại.'
)

faces = app.get(source_img)
assert len(faces) > 0, 'Không tìm thấy khuôn mặt trong ảnh nguồn, thử ảnh khác rõ mặt hơn.'
if len(faces) > 1:
    # Thứ tự app.get() trả về KHÔNG xác định -> không được lấy faces[0].
    # Ảnh nguồn có nhiều mặt thì lấy mặt to nhất (chủ thể của ảnh).
    print(f'  (ảnh nguồn có {len(faces)} mặt, dùng mặt lớn nhất)')
source_face = max(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))

print('Đã detect khuôn mặt nguồn thành công.')

## 5. Làm nét mặt (GFPGAN) — cả mục này tùy thuộc `USE_GFPGAN`

Nếu bạn đặt `USE_GFPGAN = False` ở mục 0 thì **cứ chạy tuần tự cả 3 cell dưới**, chúng sẽ tự
in một dòng rồi bỏ qua. Không cần nhớ bỏ cell nào.

### Fix riêng cho `basicsr` (bug với `torchvision` mới trên Colab)

Bản `torchvision` mới đã xoá hẳn module `torchvision.transforms.functional_tensor`, trong khi
`basicsr` vẫn import `rgb_to_grayscale` theo đường cũ đó → `ModuleNotFoundError`. Hàm này nay
nằm ở `torchvision.transforms.functional`.

Cell dưới định vị package bằng `importlib.util.find_spec` — hàm này chỉ **tìm** package chứ
không chạy `__init__.py`, nên không dính đúng cái lỗi import mà ta đang muốn vá — rồi sửa mọi
file `.py` còn tham chiếu module cũ (trong cả `basicsr`, `facexlib`, `gfpgan`).

Nó cũng tự **xoá các module hỏng khỏi `sys.modules`** cả trước lẫn sau khi vá, nhờ vậy
**không cần Runtime > Restart session**.

In [ ]:
import importlib, importlib.util, sys, pathlib

OLD_MOD = 'torchvision.transforms.functional_tensor'
NEW_MOD = 'torchvision.transforms.functional'
PKGS = ('basicsr', 'facexlib', 'gfpgan')


def purge_modules():
    """Xoá các module (có thể đã import hỏng) khỏi cache.

    Phải chạy TRƯỚC find_spec: một lần import hỏng trước đó có thể để lại
    sys.modules['basicsr'] với __spec__ = None, khiến find_spec ném ValueError
    chứ không trả về None -> cell báo nhầm "chưa cài" dù basicsr đang có trên đĩa.
    """
    for mod in [m for m in list(sys.modules) if m.split('.')[0] in PKGS]:
        del sys.modules[mod]
    importlib.invalidate_caches()


def package_dir(name):
    """Trả về thư mục package mà KHÔNG import nó.

    find_spec() chỉ định vị package, không chạy __init__.py -> không dính đúng cái
    ModuleNotFoundError mà ta đang muốn vá.
    """
    try:
        spec = importlib.util.find_spec(name)
    except Exception as e:
        print(f'  (không tra được {name}: {type(e).__name__}: {e})')
        return None
    if spec is None or not spec.submodule_search_locations:
        return None
    return pathlib.Path(list(spec.submodule_search_locations)[0])


def patch_torchvision_refs():
    purge_modules()

    targets = {}
    for name in PKGS:
        d = package_dir(name)
        if d is None:
            print(f'{name:9s}: CHƯA CÀI')
        else:
            print(f'{name:9s}: {d}')
            targets[name] = d

    assert 'basicsr' in targets, (
        'Không tìm thấy basicsr. Chạy lại cell cài basicsr ở mục 1 rồi chạy lại cell này.'
    )

    patched = []
    for name, d in targets.items():
        for p in d.rglob('*.py'):
            try:
                text = p.read_text(encoding='utf-8')
            except (UnicodeDecodeError, OSError):
                continue
            if OLD_MOD not in text:
                continue
            p.write_text(text.replace(OLD_MOD, NEW_MOD), encoding='utf-8')
            patched.append(p)

    print()
    if patched:
        for p in patched:
            print(f'đã vá: {p}')
    else:
        print('Không file nào cần vá (đã vá trước đó, hoặc torchvision bản này vẫn còn functional_tensor).')

    # Purge lần nữa sau khi vá, để lần import sau đọc lại file mới trên đĩa.
    purge_modules()

    # Kiểm chứng ngay tại đây thay vì để tới cell import gfpgan mới biết.
    try:
        import basicsr.data.degradations
        print()
        print('OK: import basicsr.data.degradations thành công.')
    except Exception as e:
        print()
        print(f'VẪN LỖI: {type(e).__name__}: {e}')
        print('Nếu lỗi vẫn liên quan tới torchvision, thử Runtime > Restart session rồi chạy lại.')
        raise


if USE_GFPGAN:
    patch_torchvision_refs()
else:
    print('USE_GFPGAN = False -> bỏ qua (không có basicsr để vá).')

In [ ]:
import subprocess, sys, importlib


def try_import_gfpgan():
    importlib.invalidate_caches()
    try:
        import gfpgan
        print('gfpgan OK, version:', getattr(gfpgan, '__version__', 'unknown'))
        return True
    except Exception as e:
        # Bắt Exception: gfpgan hỏng vì basicsr/torchvision thường ném ModuleNotFoundError,
        # nhưng tuỳ phiên bản torch cũng có thể là AttributeError/OSError.
        print(f'Chưa import được gfpgan: {type(e).__name__}: {e}')
        return False


GFPGAN_AVAILABLE = False

if not USE_GFPGAN:
    print('USE_GFPGAN = False -> bỏ qua kiểm tra gfpgan.')
else:
    GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print('Thử cài lại gfpgan với log đầy đủ...')
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'gfpgan'],
                           capture_output=True, text=True)
        print(r.stdout[-3000:])
        print(r.stderr[-3000:])
        GFPGAN_AVAILABLE = try_import_gfpgan()

    if not GFPGAN_AVAILABLE:
        print()
        print('gfpgan không cài được -> sẽ tự động chạy tiếp mà KHÔNG làm nét.')
        print('Pipeline chính (swap mặt) vẫn chạy bình thường.')

In [ ]:
# restorer = None nghĩa là "không làm nét". Vòng lặp ở mục 6 chỉ nhìn biến này,
# nên không cần kiểm tra USE_GFPGAN lần nữa ở trong đó.
restorer = None

if USE_GFPGAN and GFPGAN_AVAILABLE:
    from gfpgan import GFPGANer
    restorer = GFPGANer(
        model_path=GFPGAN_PATH,
        upscale=1,
        arch='clean',
        channel_multiplier=2,
        bg_upsampler=None
    )
    print('GFPGAN sẵn sàng (chỉ chạy trên vùng crop quanh mặt đã swap).')
elif USE_GFPGAN:
    print('USE_GFPGAN = True nhưng gfpgan không dùng được -> chạy tiếp, chỉ swap không làm nét.')
else:
    print('USE_GFPGAN = False -> chỉ swap, không làm nét.')

## 5b. Nhuộm tóc — segment + đổi màu trong LAB

Cả mục này phụ thuộc `USE_HAIR`. Tắt thì hai cell dưới in một dòng rồi bỏ qua.

### Mỗi frame chỉ có ba việc

**1. Segment** — MediaPipe trên **cả khung hình**, luôn ở 256×256. Chi phí segment là **cố
định**, không phụ thuộc độ phân giải video, nên không cần cắt hộp ROI quanh đầu như bản thay
kiểu tóc — và vì thế **không có chuyện tóc dài ra ngoài hộp thì bị bỏ sót, đổi màu nửa vời**.

Mask thô được xử lý ba bước, mỗi bước chữa một lỗi cụ thể:
- **Trừ vùng da mặt** (`HAIR_PROTECT_FACE`): mask tóc hay liếm sang trán và lông mày; không trừ
  thì nhuộm lem lên da, nhìn ra ngay.
- **Dốc mềm thay cho ngưỡng cứng** (`HAIR_CONF`): cắt cứng ở 0.5 cho ra viền răng cưa nhấp nháy
  giữa các frame, vì xác suất ở rìa dao động quanh ngưỡng.
- **EMA theo thời gian** (`HAIR_MASK_SMOOTH`): ở đây hai mask liên tiếp phủ **đúng cùng một vùng
  ảnh** (cả khung, cùng lưới 256×256) nên EMA khớp pixel-với-pixel — sạch hơn hẳn bản thay kiểu
  tóc, nơi hộp ROI dịch chuyển theo đầu nên EMA luôn lệch một chút.

**2. Đổi màu trong LAB.** Chỉ chạy trong **hộp bao quanh vùng tóc** (lấy từ chính mask), không
phải cả khung — tóc thường chiếm một phần nhỏ khung hình nên đây là chỗ tiết kiệm chính.

```
L' = mL + (L - mL)·HAIR_CONTRAST + (L_đích - mL)·HAIR_LIGHTNESS
a' = a_đích + (a - mA)·HAIR_KEEP_TONE
b' = b_đích + (b - mB)·HAIR_KEEP_TONE
```

Điểm mấu chốt nằm ở `(L - mL)`: đó chính là **sợi tóc, bóng đổ, nếp tóc**. Công thức chỉ **dịch**
cả phân bố độ sáng lên/xuống chứ không đè nó — xoá mất `(L - mL)` là ra một mảng màu phẳng lì
dán lên đầu, lỗi kinh điển của cách nhuộm bằng tô đè màu.

Mọi thống kê (`mL`, `mA`, `mB`, độ lệch chuẩn) đều **có trọng số theo mask**, nên pixel nền lọt
vào trong hộp không kéo lệch được trung bình.

**3. Chừa vùng bắt sáng.** Ánh sáng phản chiếu trên tóc thật gần như **không màu** — nhuộm cả
vào đó thì tóc mất độ bóng, trông như tóc giả. Ngưỡng "thế nào là chỗ bắt sáng" tính theo phân
bố độ sáng **của chính mái tóc trong frame đó** (`mL + sL`), không phải một hằng số, nên tóc
sáng/tối, cảnh ngược sáng hay thiếu sáng đều tự khớp. Độ sáng vẫn được nhuộm ở vùng này, chỉ
riêng màu là chừa lại.

### Vì sao không phải HSV, và không phải "tô đè rồi giảm opacity"

- **Tô đè + opacity**: pha thẳng màu đặc lên pixel gốc thì kéo tụt tương phản của cả vùng tóc,
  cho ra mảng bệt. Ở LAB, độ sáng và màu sắc tách rời nên đổi được màu mà **không đụng vào**
  cấu trúc sáng-tối.
- **HSV**: kênh `V` không phải độ sáng cảm nhận được (`V` của xanh dương thẫm và của vàng chói
  bằng nhau), nên chỉnh `H` ở vùng tối sẽ nhảy màu loang lổ. `L` của LAB gần với cảm nhận mắt
  người hơn nhiều.

### Giới hạn vật lý

Tóc gần đen có `L` rất thấp **và** biến thiên `(L - mL)` rất nhỏ — không còn chi tiết để giữ.
Nhuộm sang màu sáng thì phải kéo `HAIR_LIGHTNESS` lên cao, và kết quả sẽ phẳng hơn tóc vốn đã
sáng màu. Ngoài đời cũng phải tẩy tóc trước mới nhuộm sáng được. `HAIR_CONTRAST` (1.2-1.4) bù
lại được một phần.

In [ ]:
import numpy as np
import cv2

# Nhãn của model selfie_multiclass_256x256 (theo tài liệu MediaPipe).
LBL_BACKGROUND, LBL_HAIR, LBL_BODY_SKIN, LBL_FACE_SKIN, LBL_CLOTHES, LBL_OTHERS = range(6)

SEG_SIZE = 256   # model vốn chạy ở 256x256; đưa vào đúng cỡ này để nó không phải nội suy 2 lần


def mask_2d(arr):
    """Ép mask của MediaPipe về đúng 2D (H, W).

    Có bản mediapipe trả confidence mask shape (H, W, 1) thay vì (H, W). Trộn hai kiểu này
    lại là lỗi ngầm rất khó đoán: cv2.GaussianBlur/resize LẶNG LẼ bỏ chiều cuối, nên một mask
    thành 2D còn mask kia vẫn 3D, rồi phép nhân giữa chúng nổ
    "non-broadcastable output operand ... doesn't match the broadcast shape (256,256,256)"
    - hoặc tệ hơn, np.nonzero() trả 3 mảng thay vì 2. Chuẩn hoá ngay tại nguồn, một lần.
    """
    # copy=True là BẮT BUỘC, không phải cho chắc: np.asarray() trên mảng float32 đã liền khối
    # trả về CHÍNH mảng đó, tức là một view vào bộ nhớ MediaPipe - và bộ nhớ đó bị ghi đè ở lần
    # segment() sau, làm mask của frame trước đổi giá trị sau lưng mình (EMA thành vô nghĩa).
    a = np.array(arr, dtype=np.float32, copy=True)
    if a.ndim == 3 and a.shape[2] == 1:
        a = a[:, :, 0]              # view vào BẢN COPY ở trên -> vẫn an toàn
    assert a.ndim == 2, f'Mask của MediaPipe có shape lạ: {a.shape}'
    return np.ascontiguousarray(a)


def hex_to_bgr(s):
    s = s.strip().lstrip('#')
    assert len(s) == 6, f'Mã màu phải dạng #RRGGBB, nhận được {s!r}'
    r, g, b = int(s[0:2], 16), int(s[2:4], 16), int(s[4:6], 16)
    return np.uint8([[[b, g, r]]])


def bgr_to_hex(bgr):
    b, g, r = [int(v) for v in bgr]
    return f'#{r:02X}{g:02X}{b:02X}'


def to_lab(bgr_1px):
    """Một pixel BGR -> (L, a, b) trong thang uint8 của OpenCV (L 0-255, a/b lệch 128)."""
    return cv2.cvtColor(bgr_1px, cv2.COLOR_BGR2LAB)[0, 0].astype(np.float32)


class HairRecolor:
    """Đổi màu tóc trong từng frame: segment -> đổi màu trong LAB -> blend.

    KHÔNG cần landmark, KHÔNG cần biết đầu quay hướng nào, KHÔNG cần warp gì cả:
    tóc đã nằm sẵn đúng chỗ trong frame, việc duy nhất là đổi màu những pixel đó.
    """

    def __init__(self, model_path):
        import mediapipe as mp
        from mediapipe.tasks import python as mp_python
        from mediapipe.tasks.python import vision as mp_vision

        self._mp = mp
        common = dict(base_options=mp_python.BaseOptions(model_asset_path=model_path),
                      running_mode=mp_vision.RunningMode.IMAGE)
        try:
            options = mp_vision.ImageSegmenterOptions(
                **common,
                output_category_mask=False,   # chỉ cần confidence mask (float, mép mềm)
                output_confidence_masks=True,
            )
        except TypeError as e:
            # mediapipe < 0.10.3 dùng bộ tham số khác. Không ghim version ở mục 1 (dễ hết wheel
            # cho Python mới trên Colab) nên chấp nhận cả hai: _segment() tự xoay theo kết quả.
            print(f'  (ImageSegmenterOptions bản cũ: {e})')
            options = mp_vision.ImageSegmenterOptions(**common)
        self.segmenter = mp_vision.ImageSegmenter.create_from_options(options)

        self.prev_masks = None      # EMA theo thời gian
        self.last_alpha = None      # vùng đã nhuộm ở frame gần nhất (để xem lại ở mục 6.1)
        self.last_box = None
        self.target_lab = None      # (L, a, b) của màu đích
        self.target_bgr = None
        self.n_applied = 0

    # ---------------------------------------------------------------- segment
    def _segment(self, bgr):
        """Trả (hair, face_skin) - hai mask xác suất float32 cỡ SEG_SIZE x SEG_SIZE.

        Luôn đưa vào đúng SEG_SIZE, và luôn là CẢ KHUNG HÌNH. Nhờ chi phí segment cố định
        (không phụ thuộc độ phân giải video) nên không cần cắt ROI quanh đầu như bản thay
        kiểu tóc -> không có chuyện tóc dài ra ngoài hộp thì bị bỏ sót, đổi màu nửa vời.
        """
        small = cv2.resize(bgr, (SEG_SIZE, SEG_SIZE), interpolation=cv2.INTER_AREA)
        rgb = np.ascontiguousarray(cv2.cvtColor(small, cv2.COLOR_BGR2RGB))
        mp_img = self._mp.Image(image_format=self._mp.ImageFormat.SRGB, data=rgb)
        res = self.segmenter.segment(mp_img)
        # copy=True: numpy_view() chỉ là view vào bộ nhớ của MediaPipe, bị ghi đè ở lần
        # segment() sau -> không copy thì mask của frame trước đổi giá trị sau lưng mình.
        masks = getattr(res, 'confidence_masks', None)
        if masks:
            # mask_2d() vừa copy vừa ép về 2D - xem docstring của nó để biết vì sao cần cả hai.
            hair = mask_2d(masks[LBL_HAIR].numpy_view())
            face = mask_2d(masks[LBL_FACE_SKIN].numpy_view())
        else:
            # Bản mediapipe không cho confidence mask -> lấy category mask (0/1, mép gắt).
            # Vẫn chạy được vì bước sau còn làm mềm mép bằng Gaussian.
            cat = mask_2d(res.category_mask.numpy_view())
            hair = (cat == LBL_HAIR).astype(np.float32)
            face = (cat == LBL_FACE_SKIN).astype(np.float32)
        return hair, face

    # ------------------------------------------------------------- chọn màu
    def hair_color_of(self, img):
        """Màu tóc trung bình của một ảnh (dùng cho HAIR_COLOR = 'ảnh'). Trả (bgr, tỉ lệ phủ).

        Lấy trung vị chứ không phải trung bình: vùng tóc thường dính vài pixel nền/da ở rìa,
        trung bình bị chúng kéo lệch, trung vị thì không.
        """
        h, w = img.shape[:2]
        hair, _ = self._segment(img)
        m = cv2.resize(hair, (w, h), interpolation=cv2.INTER_LINEAR) >= 0.6
        cover = float(m.mean())
        assert m.sum() > 200, (
            'Không tìm thấy vùng tóc trong ảnh nguồn để lấy màu.\n'
            'Dùng ảnh khác thấy rõ tóc, hoặc đặt HAIR_COLOR bằng tên màu / mã hex.'
        )
        return np.median(img[m], axis=0).astype(np.uint8), cover

    def set_target(self, bgr):
        bgr = np.asarray(bgr, dtype=np.uint8).reshape(1, 1, 3)
        self.target_bgr = bgr[0, 0].copy()
        self.target_lab = to_lab(bgr)

    def reset(self):
        self.prev_masks = None

    # ---------------------------------------------------------- tóc mai
    @staticmethod
    def _wisp_matte(L, cov, kps_local, d, L_hair):
        """Bắt những SỢI TÓC MAI mảnh rủ trước mặt, thứ mà mask 256x256 không thấy nổi.

        Sợi tóc mai rộng cỡ 4-8 pixel trên video 1080p, tức CHƯA TỚI MỘT Ô của mask 256x256 -
        ô đó bị da mặt phía sau lấn át nên xác suất "tóc" rất thấp. Tệ hơn, HAIR_PROTECT_FACE
        còn trừ thẳng vùng da mặt ra khỏi mask, mà tóc mai nằm đúng trên đó. Kết quả: cả mái
        tóc đổi màu, riêng mấy sợi trước mặt vẫn đen.

        Cách bắt: sợi tóc mai là VẠCH TỐI MẢNH trên nền da sáng hơn. Phép "black-hat" (đóng
        ảnh rồi trừ đi ảnh gốc) cho ra đúng những vạch tối mảnh hơn hạt nhân, và giá trị của
        nó chính là ĐỘ CHÊNH SÁNG giữa sợi tóc và da xung quanh. Chia cho chênh lệch tối đa
        (da - tóc) thì ra luôn tỉ lệ phủ của sợi trên pixel đó - cùng một nguyên tắc với phần
        xử lý mép tóc.

        Vùng mắt/mũi/miệng bị loại trừ: chúng cũng là "vạch tối trên nền da", nhuộm vào đó thì
        hỏng mặt. Lông mày nằm trong vùng loại trừ luôn, vì nhuộm lông mày hiếm khi là ý muốn.
        """
        h, w = cov.shape
        k = int(np.clip(0.18 * d, 9, 41)) | 1
        closing = cv2.morphologyEx(L, cv2.MORPH_CLOSE,
                                   cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        blackhat = closing - L                    # vạch càng tối so với xung quanh thì càng lớn

        # Tỉ lệ phủ = chênh sáng đo được / chênh sáng tối đa có thể (da xung quanh - tóc).
        # Trừ 6 làm ngưỡng nhiễu, nếu không thì vân da và nhiễu ảnh cũng bị nhận là tóc.
        denom = np.maximum(closing - L_hair, 15.0)
        wisp = np.clip((blackhat - 6.0) / denom, 0.0, 1.0)

        # Chỉ nhận sợi ở GẦN mái tóc, tránh nhuộm nhầm vạch tối chỗ khác (viền áo, đồ vật).
        # Bán kính phải rộng (hơn một khoảng cách hai mắt): tóc mai rủ dài xuống quá cằm, lấy
        # hẹp thì chỉ nhuộm được đoạn đầu của sợi rồi cụt giữa chừng - còn xấu hơn để nguyên.
        # Hạt nhân CHỮ NHẬT vì OpenCV làm nó tách rời theo hai trục, bán kính lớn vẫn rẻ.
        solid = (cov >= 0.5).astype(np.uint8)
        rn = max(3, int(2.0 * d))
        near = cv2.dilate(solid, cv2.getStructuringElement(cv2.MORPH_RECT, (2 * rn + 1, 2 * rn + 1)))
        wisp = wisp * cv2.GaussianBlur(near.astype(np.float32), (0, 0), sigmaX=max(1.0, 0.05 * d))

        # Loại trừ mắt/mũi/miệng/lông mày: hình bầu dục xoay theo trục hai mắt, nên đầu nghiêng
        # vẫn che đúng chỗ.
        le, re = kps_local[0], kps_local[1]
        mouth = (kps_local[3] + kps_local[4]) / 2.0
        cen = ((le + re) / 2.0 + mouth) / 2.0
        ang = float(np.degrees(np.arctan2(re[1] - le[1], re[0] - le[0])))
        feat = np.zeros((h, w), np.float32)
        cv2.ellipse(feat, (int(round(cen[0])), int(round(cen[1]))),
                    (int(1.05 * d), int(1.2 * d)), ang, 0, 360, 1.0, -1)
        feat = cv2.GaussianBlur(feat, (0, 0), sigmaX=max(1.0, 0.12 * d))
        return wisp * (1.0 - feat)

    # ------------------------------------------------------------ mép tóc
    @staticmethod
    def _edge_matte(L, cov, edge_px):
        """Ước lượng lại độ phủ tóc ở DẢI MÉP bằng chính độ sáng của ảnh.

        Vì sao cần: mask của MediaPipe chỉ có 256x256. Phóng lên video 1080p thì mỗi ô mask
        thành hơn 4 pixel ảnh, và trong dải mép rộng cỡ đó mask KHÔNG biết pixel nào là tóc,
        pixel nào là nền - nó chỉ trả về một con số lưng chừng. Chỉ dựa vào mask thì có đúng
        hai lựa chọn, cả hai đều tệ:
          - nới mask rộng ra  -> nền quanh đầu bị nhuộm lây, thành QUẦNG SÁNG viền tóc;
          - co mask hẹp lại   -> mép tóc thật không được nhuộm, thành VIỀN TỐI màu tóc cũ.
        Chỉnh qua chỉnh lại giữa hai cái đó không bao giờ ra kết quả sạch, vì thiếu thông tin.

        Nhưng chính bức ảnh có thông tin đó: pixel ở mép là màu PHA giữa tóc và nền, nên vị
        trí của nó trên thang độ sáng (nền <-> tóc) chính là TỈ LỆ PHA - tức độ phủ cần tìm.
        Nền được ước lượng CỤC BỘ quanh từng điểm chứ không lấy một giá trị chung cho cả ảnh,
        vì quanh đầu chỗ là tường, chỗ là vai áo, chỗ là rèm.
        """
        h, wd = cov.shape
        core = (cov >= 0.9).astype(np.float32)
        n_core = float(core.sum())
        if n_core < 50:
            return cov              # tóc quá nhỏ/mảnh -> không đủ cơ sở, cứ tin mask

        L_hair = float((L * core).sum() / n_core)

        # Độ sáng NỀN cục bộ = trung bình L của những pixel chắc chắn không phải tóc, trong một
        # cửa sổ quanh mỗi điểm (tích chập chuẩn hoá: den là số mẫu rơi vào cửa sổ đó).
        k = max(9, (int(0.15 * max(h, wd)) | 1))
        bg = (cov < 0.05).astype(np.float32)
        den = cv2.blur(bg, (k, k))
        L_bg = cv2.blur(L * bg, (k, k)) / np.maximum(den, 1e-3)

        d = L_hair - L_bg
        ok = np.abs(d) > 12.0       # tóc và nền sáng xấp xỉ nhau -> không tách được bằng độ sáng
        key = np.clip((L - L_bg) / np.where(ok, d, 1.0), 0.0, 1.0)
        key = np.where(ok, key, cov)                 # không tách được thì tin mask như cũ

        # Key CHỈ được dùng ở VÀNH NGOÀI - vùng thật sự lẫn tóc với nền. Sâu bên trong mái
        # tóc thì nhuộm đủ, KHÔNG hỏi key: chỗ tóc bắt sáng sáng gần bằng nền nên key ở đó
        # thấp, nghe theo nó là để lại nguyên mảng màu cũ ngay giữa đỉnh đầu và chỗ rẽ ngôi.
        # "Sâu bên trong" = co vùng mask vào edge_px pixel, chứ không phải "cov cao": mask
        # hay lưỡng lự (0.5-0.7) ngay giữa mái tóc, mà chỗ đó vẫn chắc chắn là tóc.
        r = max(1, int(edge_px))
        solid = (cov >= 0.5).astype(np.uint8)
        deep = cv2.erode(solid, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
        deep = cv2.blur(deep.astype(np.float32), (r | 1, r | 1))     # làm mềm bậc thang

        core_w = np.clip((cov - 0.85) / 0.15, 0.0, 1.0)
        support = np.clip(cov / 0.10, 0.0, 1.0)                      # chặn key lan ra ngoài mask
        return np.clip(np.maximum(deep, support * np.maximum(key, core_w)), 0.0, 1.0)

    # ---------------------------------------------------------------- nhuộm
    def apply(self, frame, kps=None):
        """Đổi màu tóc trong frame (ghi trực tiếp vào frame). Trả (frame, đã_nhuộm?).

        kps = 5 điểm mốc khuôn mặt, CHỈ dùng cho phần tóc mai (để biết chỗ nào là mắt/mũi/
        miệng mà tránh ra). Không truyền cũng chạy bình thường, chỉ là bỏ qua tóc mai.
        """
        assert self.target_lab is not None, 'Chưa gọi set_target() để chọn màu.'
        H, W = frame.shape[:2]

        hair, face = self._segment(frame)
        if self.prev_masks is not None and HAIR_MASK_SMOOTH > 0:
            # EMA thẳng trong không gian 256x256 của CẢ KHUNG: khác bản thay kiểu tóc, ở đây
            # hai mask liên tiếp phủ đúng cùng một vùng ảnh nên EMA khớp pixel-với-pixel,
            # không có sai lệch do hộp ROI dịch chuyển.
            hair = HAIR_MASK_SMOOTH * self.prev_masks[0] + (1 - HAIR_MASK_SMOOTH) * hair
            face = HAIR_MASK_SMOOTH * self.prev_masks[1] + (1 - HAIR_MASK_SMOOTH) * face
        self.prev_masks = (hair, face)

        lo, hi = HAIR_CONF
        m = np.clip((hair - lo) / max(hi - lo, 1e-6), 0.0, 1.0)
        r = int(round(HAIR_EDGE_CHOKE))
        if r >= 1:
            # CO mask vào trong TRƯỚC khi làm mềm. Không có bước này thì phần nhoè của
            # GaussianBlur tràn ra nền quanh đầu, và với cú nhảy độ sáng lớn (đen -> trắng,
            # +190 L) thì alpha 0.2 ở đó cũng đủ vẽ ra một quầng sáng viền tóc rất lộ.
            # Co trước - làm mềm sau => toàn bộ dải chuyển tiếp nằm bên trong tóc.
            m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
        if HAIR_FEATHER > 0:
            # Làm mềm mép ngay ở không gian 256: rẻ hơn nhiều so với blur trên frame gốc,
            # và vì tính theo tỉ lệ nên độ mềm tự co giãn theo độ phân giải video.
            m = cv2.GaussianBlur(m, (0, 0), sigmaX=HAIR_FEATHER)
        if HAIR_PROTECT_FACE:
            # Trừ da mặt SAU khi làm mềm mép, không phải trước: làm mềm trước rồi trừ sau thì
            # chính cái blur lại kéo mask tràn ngược vào trán vài pixel -> da bị ám màu ở chân
            # tóc. Mask da mặt của MediaPipe vốn đã mềm sẵn nên chỗ này không thành mép cứng.
            m *= (1.0 - face)

        ys, xs = np.nonzero(m > 0.02)
        if len(ys) < 20:
            return frame, False        # frame không có tóc (quay xa, che khuất...)

        # Hộp bao quanh vùng tóc, quy về toạ độ frame. Phần đổi màu (chuyển LAB, trộn) chỉ
        # chạy trong hộp này - đó là chỗ tiết kiệm chính, vì tóc thường chiếm ít khung hình.
        sx, sy = W / SEG_SIZE, H / SEG_SIZE
        mx1, mx2 = int(xs.min()), int(xs.max()) + 1
        my1, my2 = int(ys.min()), int(ys.max()) + 1

        if HAIR_WISPS > 0 and kps is not None:
            # Nới hộp ra bao cả khuôn mặt. Hộp mặc định là hộp bao của MÁI TÓC, mà tóc mai thì
            # rủ TRƯỚC MẶT - nằm ngoài hộp đó, nên không nới thì tầng tóc mai không có gì để
            # nhìn (mọi thứ ngoài hộp đều không được đụng tới).
            k256 = np.asarray(kps, dtype=np.float64) / np.array([sx, sy])
            d256 = float(np.linalg.norm(k256[1] - k256[0]))
            cx, cy = k256.mean(0)
            mx1 = max(0, min(mx1, int(np.floor(cx - 2.2 * d256))))
            mx2 = min(SEG_SIZE, max(mx2, int(np.ceil(cx + 2.2 * d256))))
            my1 = max(0, min(my1, int(np.floor(cy - 2.0 * d256))))
            my2 = min(SEG_SIZE, max(my2, int(np.ceil(cy + 3.0 * d256))))

        x1 = max(0, int(np.floor(mx1 * sx))); x2 = min(W, int(np.ceil(mx2 * sx)))
        y1 = max(0, int(np.floor(my1 * sy))); y2 = min(H, int(np.ceil(my2 * sy)))
        if x2 - x1 < 4 or y2 - y1 < 4:
            return frame, False

        crop = frame[y1:y2, x1:x2]
        sub = m[my1:my2, mx1:mx2]
        cov = np.clip(cv2.resize(sub, (x2 - x1, y2 - y1), interpolation=cv2.INTER_LINEAR), 0.0, 1.0)

        lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).astype(np.float32)
        L, A, B = lab[..., 0], lab[..., 1], lab[..., 2]
        Lt, At, Bt = self.target_lab

        if HAIR_EDGE_SMART:
            # Bề rộng vành ngoài, quy ra pixel ảnh: mask 256x256 phóng lên video nên một ô mask
            # bằng (H+W)/2/256 pixel ảnh; lấy ~3 ô cho đủ phần mask lưỡng lự + phần làm mềm mép.
            edge_px = max(3, int(round(3.0 * (H + W) / 2.0 / SEG_SIZE)))
            cov = self._edge_matte(L, cov, edge_px)

        if HAIR_WISPS > 0 and kps is not None:
            kl = np.asarray(kps, dtype=np.float64) - np.array([x1, y1], dtype=np.float64)
            d_eye = float(np.linalg.norm(kl[1] - kl[0]))
            core = cov >= 0.9
            if d_eye > 8 and core.sum() >= 50:
                wisp = self._wisp_matte(L, cov, kl, d_eye, float(L[core].mean()))
                cov = np.maximum(cov, HAIR_WISPS * wisp)

        # Giữ lại để cell 6.1 vẽ ra ĐÚNG vùng đã nhuộm (xem thẳng còn hơn đoán).
        self.last_alpha, self.last_box = cov, (x1, y1, x2, y2)

        w = cov * HAIR_STRENGTH
        wsum = float(w.sum())
        if wsum < 20:
            return frame, False

        # Thống kê CÓ TRỌNG SỐ theo mask: chỉ pixel tóc mới được tính, pixel nền lọt vào hộp
        # không được kéo lệch trung bình.
        mL = float((L * w).sum() / wsum)
        mA = float((A * w).sum() / wsum)
        mB = float((B * w).sum() / wsum)
        sL = float(np.sqrt(max(((L - mL) ** 2 * w).sum() / wsum, 1e-6)))

        # --- ĐỘ SÁNG: dịch cả phân bố, GIỮ NGUYÊN biến thiên quanh trung bình ---
        # (L - mL) chính là sợi tóc / bóng đổ / nếp tóc. Xoá nó đi là ra mảng màu phẳng lì.
        L_new = mL + (L - mL) * HAIR_CONTRAST + (Lt - mL) * HAIR_LIGHTNESS

        # --- MÀU: thay a/b bằng màu đích, giữ lại một phần biến thiên gốc ---
        A_new = At + (A - mA) * HAIR_KEEP_TONE
        B_new = Bt + (B - mB) * HAIR_KEEP_TONE
        if HAIR_SATURATION != 1.0:
            A_new = 128.0 + (A_new - 128.0) * HAIR_SATURATION
            B_new = 128.0 + (B_new - 128.0) * HAIR_SATURATION

        # --- Chừa vùng bắt sáng: phản chiếu trên tóc thật gần như không màu ---
        # Ngưỡng tính theo phân bố độ sáng CỦA CHÍNH MÁI TÓC NÀY (mL, sL), không phải hằng số,
        # nên tóc tối hay sáng, cảnh ngược sáng hay thiếu sáng đều tự khớp.
        wc = w
        if HAIR_KEEP_HIGHLIGHT > 0:
            # Bắt đầu chừa từ 1.5 lần độ lệch chuẩn trở lên, đầy đủ từ 3.0. Đặt ngưỡng thấp hơn
            # (1 sd) sẽ chạm cả những sợi tóc sáng bình thường - khoảng 16% số pixel - làm tóc
            # ăn màu loang lổ. Phản chiếu thật thì sáng vượt hẳn, tầm 2 sd trở lên.
            hl = np.clip((L - (mL + 1.5 * sL)) / max(1.5 * sL, 1e-6), 0.0, 1.0)
            wc = w * (1.0 - hl * HAIR_KEEP_HIGHLIGHT)

        lab[..., 0] = L + (L_new - L) * w        # độ sáng: nhuộm cả vùng bóng
        lab[..., 1] = A + (A_new - A) * wc       # màu: chừa vùng bóng lại
        lab[..., 2] = B + (B_new - B) * wc

        np.clip(lab, 0, 255, out=lab)
        frame[y1:y2, x1:x2] = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2BGR)
        self.n_applied += 1
        return frame, True


recolor = None

if not USE_HAIR:
    print('USE_HAIR = False -> giữ nguyên màu tóc trong video.')
elif not HAIR_AVAILABLE:
    print('USE_HAIR = True nhưng mediapipe không dùng được -> chạy tiếp, chỉ swap mặt.')
else:
    recolor = HairRecolor(SEGMENTER_PATH)

    if HAIR_COLOR == 'ảnh':
        bgr, cover = recolor.hair_color_of(source_img)
        print(f'Màu tóc lấy từ ảnh nguồn: {bgr_to_hex(bgr)}  '
              f'(vùng tóc chiếm {cover:.1%} ảnh)')
        if cover < 0.02:
            print('  (vùng tóc khá nhỏ trong ảnh -> màu lấy được có thể không đại diện)')
    else:
        hx = HAIR_PALETTE.get(HAIR_COLOR, HAIR_COLOR)
        bgr = hex_to_bgr(hx)[0, 0]
        print(f'Màu tóc: {HAIR_COLOR} = {bgr_to_hex(bgr)}')

    recolor.set_target(bgr)
    L, A, B = recolor.target_lab
    print(f'  LAB đích: L={L:.0f}/255  a={A - 128:+.0f}  b={B - 128:+.0f}')
    print('Sẵn sàng nhuộm tóc.')

In [ ]:
# Xem mask tóc + bảng màu TRƯỚC khi chạy cả video. Mask sai ở đây thì mọi frame đều sai.
import numpy as np
import cv2

if recolor is None:
    print('Không có recolor -> không có gì để xem.')
else:
    from google.colab.patches import cv2_imshow

    # ---- 1. mask tóc trên chính ảnh nguồn (kiểm tra segment có ăn không) ----
    h, w = source_img.shape[:2]
    hair, face = recolor._segment(source_img)
    lo, hi = HAIR_CONF
    m = np.clip((hair - lo) / max(hi - lo, 1e-6), 0.0, 1.0)   # cùng thứ tự với apply():
    r = int(round(HAIR_EDGE_CHOKE))                           # ngưỡng -> co -> làm mềm
    if r >= 1:                                                #   -> trừ da mặt
        m = cv2.erode(m, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1)))
    if HAIR_FEATHER > 0:
        m = cv2.GaussianBlur(m, (0, 0), sigmaX=HAIR_FEATHER)
    if HAIR_PROTECT_FACE:
        m *= (1.0 - face)
    m = cv2.resize(m, (w, h), interpolation=cv2.INTER_LINEAR)[..., None]

    red = np.zeros_like(source_img); red[..., 2] = 255
    overlay = (source_img * (1 - 0.55 * m) + red * (0.55 * m)).astype(np.uint8)
    strip = np.hstack([source_img, overlay])
    sc = min(1.0, 900 / strip.shape[1])
    if sc < 1.0:
        strip = cv2.resize(strip, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
    print('Mask tóc trên ảnh nguồn (đỏ = sẽ bị nhuộm):')
    cv2_imshow(strip)
    print('  lem sang nền/da  -> nâng HAIR_CONF, vd (0.5, 0.8)')
    print('  hụt rìa tóc      -> hạ HAIR_CONF, vd (0.25, 0.5)')
    print()

    # ---- 2. bảng màu + màu đang chọn ----
    names = list(HAIR_PALETTE)
    cols, cell_w, cell_h = 6, 150, 70
    rows = (len(names) + cols - 1) // cols
    board = np.full((rows * cell_h, cols * cell_w, 3), 255, np.uint8)
    for i, nm in enumerate(names):
        r, c = divmod(i, cols)
        y, x = r * cell_h, c * cell_w
        bgr = hex_to_bgr(HAIR_PALETTE[nm])[0, 0]
        cv2.rectangle(board, (x + 4, y + 4), (x + cell_w - 4, y + cell_h - 22),
                      [int(v) for v in bgr], -1)
        if nm == HAIR_COLOR:      # khoanh viền màu đang chọn
            cv2.rectangle(board, (x + 2, y + 2), (x + cell_w - 2, y + cell_h - 20), (0, 0, 255), 2)
        # cv2.putText không vẽ được dấu tiếng Việt -> ghi chỉ số, tên in ra bằng print ở dưới.
        cv2.putText(board, str(i), (x + 8, y + cell_h - 6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
    print('Bảng màu (viền đỏ = đang chọn):')
    cv2_imshow(board)
    print('  ' + ' | '.join(f'{i}={nm}' for i, nm in enumerate(names)))
    print()

    # ---- 3. màu đích đang dùng ----
    sw = np.zeros((80, 300, 3), np.uint8)
    sw[:] = [int(v) for v in recolor.target_bgr]
    print(f'Màu đích đang dùng: {bgr_to_hex(recolor.target_bgr)}')
    cv2_imshow(sw)

## 6. Xử lý video

Mỗi frame:

1. `app.get(frame)` → detect mặt, `pick_face(...)` → chọn **mặt lớn nhất**
2. `swapper.get(..., paste_back=True)` → swap mặt
3. `enhance_face_region(...)` → làm nét, chỉ khi `restorer is not None`
4. `recolor.apply(...)` → **nhuộm tóc**, chỉ khi `recolor is not None`

**Bước 1-3 nằm trong nhánh "có detect được mặt". Bước 4 thì không** — nhuộm màu không cần
landmark, nên frame nào người quay lưng / cúi đầu / mặt bị che thì swap bỏ qua, **tóc vẫn đổi
màu bình thường**. Dòng thống kê cuối mục 6.2 in riêng số frame nhuộm được mà không thấy mặt.

**Bước 4 phải nằm sau bước 3**: ô crop của GFPGAN nới bbox thêm 40% nên lấn sang cả tóc; nhuộm
trước thì GFPGAN vẽ lại chính vùng tóc vừa nhuộm, mỗi frame một kiểu → nhấp nháy.

Mục này chia làm **bốn cell**:

- **6.0** định nghĩa `process_frame()` — dùng chung cho cả preview lẫn vòng lặp thật, nên không
  có chuyện preview chạy một đường mà video ra một nẻo.
- **6.1** chạy **đúng một frame**, in ảnh trước/sau.
- **6.1b** thử **nhiều màu cùng lúc** trên một frame, xếp thành lưới để chọn bằng mắt. Phần
  swap mặt chỉ chạy một lần rồi dùng lại cho mọi màu.
- **6.2** chạy toàn bộ video, ghi thẳng vào `ffmpeg` qua pipe và mux audio trong cùng một pass.

In [ ]:
import os, sys, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

MIN_DET_SCORE = 0.5      # bỏ qua detect yếu (thường là false positive trong nền)
GFPGAN_PAD    = 0.4      # nới bbox bao nhiêu lần khi crop để làm nét
final_output  = '/content/output_final.mp4'

# globals().get: cho phép bỏ qua HẲN mục 5 / 5b (không chạy cell nào ở đó) mà vẫn chạy được
# pipeline chính, thay vì NameError.
restorer = globals().get('restorer', None)
recolor  = globals().get('recolor', None)
print('Làm nét mặt:', 'BẬT' if restorer is not None else 'TẮT')
print('Nhuộm tóc  :', 'BẬT' if recolor is not None else 'TẮT')


def pick_face(faces):
    """Chọn khuôn mặt để swap trong một frame. Trả về None nếu không có mặt nào đủ tốt.

    Video 1 người nên không cần tracking theo danh tính: mặt LỚN NHẤT là chủ thể.
    KHÔNG dùng faces[0] vì thứ tự app.get() trả về không xác định.
    """
    cands = [f for f in faces if f.det_score >= MIN_DET_SCORE]
    if not cands:
        return None
    return max(cands, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))


def enhance_face_region(img, face, pad=GFPGAN_PAD):
    """Chỉ làm nét vùng quanh khuôn mặt vừa swap, KHÔNG chạy trên cả frame.

    Gọi restorer.enhance() trên nguyên frame khiến GFPGAN chạy lại một bộ face detector
    thứ hai trên toàn khung (trùng lặp với app.get() vừa chạy), làm nét luôn cả những mặt
    trong nền không hề bị swap, và resize LANCZOS toàn frame mỗi lượt.
    """
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face.bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = max(0, int(x1 - pad * bw)); x2 = min(w, int(x2 + pad * bw))
    y1 = max(0, int(y1 - pad * bh)); y2 = min(h, int(y2 + pad * bh))
    cw, ch = x2 - x1, y2 - y1
    if cw < 64 or ch < 64:
        return img

    crop = img[y1:y2, x1:x2]
    _, _, restored = restorer.enhance(crop, has_aligned=False,
                                      only_center_face=True, paste_back=True)
    if restored is None:
        return img
    if restored.shape[:2] != (ch, cw):
        restored = cv2.resize(restored, (cw, ch), interpolation=cv2.INTER_LANCZOS4)

    # Blend viền mềm để vùng crop không để lại đường viền hình chữ nhật thấy rõ.
    mask = np.zeros((ch, cw), np.float32)
    cv2.rectangle(mask, (int(0.12 * cw), int(0.12 * ch)),
                  (int(0.88 * cw), int(0.88 * ch)), 1.0, -1)
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(2.0, 0.05 * max(cw, ch)))[..., None]
    img[y1:y2, x1:x2] = (restored * mask + crop * (1.0 - mask)).astype(np.uint8)
    return img


STATS = dict(frames=0, swapped=0, haired=0, detect=0.0, swap=0.0, enhance=0.0, hair=0.0)


def process_frame(frame):
    """Xử lý MỘT frame. Dùng chung cho cả cell thử 1 frame lẫn vòng lặp cả video.

    Thứ tự: detect -> swap -> làm nét -> NHUỘM TÓC.
    Nhuộm phải là bước cuối: ô crop của GFPGAN (pad 0.4) lấn sang cả tóc, nhuộm trước thì
    GFPGAN sẽ vẽ lại chính vùng tóc vừa nhuộm, mỗi frame một kiểu -> nhấp nháy.

    Khác bản thay kiểu tóc: phần tóc KHÔNG nằm trong nhánh 'có mặt'. Nhuộm màu không cần
    landmark, nên frame nào người quay lưng / cúi đầu / mặt bị che - swap bỏ qua nhưng tóc
    vẫn đổi màu bình thường.
    """
    out = frame

    t0 = time.perf_counter()
    target_face = pick_face(app.get(frame))
    t1 = time.perf_counter()
    STATS['detect'] += t1 - t0

    if target_face is not None:
        # swapper.get(paste_back=True) trả về mảng MỚI, nên các bước sau ghi đè tại chỗ đều an toàn.
        out = swapper.get(frame, target_face, source_face, paste_back=True)
        t2 = time.perf_counter()
        STATS['swap'] += t2 - t1

        if restorer is not None:
            out = enhance_face_region(out, target_face)
        STATS['enhance'] += time.perf_counter() - t2
        STATS['swapped'] += 1

    t3 = time.perf_counter()
    if recolor is not None:
        # Truyền landmark (nếu có) để phần tóc mai biết chừa mắt/mũi/miệng ra.
        # Frame không có mặt vẫn nhuộm bình thường, chỉ bỏ qua phần tóc mai.
        out, applied = recolor.apply(out, target_face.kps if target_face is not None else None)
        STATS['haired'] += int(applied)
    STATS['hair'] += time.perf_counter() - t3

    STATS['frames'] += 1
    return out


def reset_state():
    """Đưa STATS và trạng thái thời gian về 0 (gọi trước mỗi lần chạy lại)."""
    for k in STATS:
        STATS[k] = 0 if isinstance(STATS[k], int) else 0.0
    if recolor is not None:
        recolor.reset()
        recolor.n_applied = 0


print('Đã định nghĩa process_frame(). Chạy cell 6.1 để thử 1 frame trước khi làm cả video.')

In [ ]:
# Thử ĐÚNG MỘT frame để soi kết quả và chỉnh màu. Vài giây một vòng, thay vì xử lý cả video.
# Chỉnh ở mục 0 -> chạy lại cell mục 0 -> chạy lại cell mục 5b (nạp màu mới) -> chạy lại cell này.
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

PREVIEW_AT = 0.5      # lấy frame ở đâu trong video: 0.0 = đầu, 0.5 = giữa, 0.95 = gần cuối

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'
n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if n_total > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n_total * PREVIEW_AT))
ok, frame = cap.read()
if not ok:
    # Seek thất bại với một số container -> quay lại đọc tuần tự từ đầu.
    cap.release()
    cap = cv2.VideoCapture(source_video_path)
    ok, frame = cap.read()
cap.release()
assert ok, 'Không đọc được frame nào từ video.'

before = frame.copy()

reset_state()
# Chạy 2 lần trên cùng 1 frame: lần đầu khởi tạo EMA mask (prev_masks = None), lần hai mới đúng
# trạng thái mà video thật sẽ có. Không làm vậy thì preview khác kết quả cuối một chút.
process_frame(frame.copy())
after = process_frame(frame.copy())

strip = np.hstack([before, after])
sc = min(1.0, 1100 / strip.shape[1])
if sc < 1.0:
    strip = cv2.resize(strip, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
print('TRƯỚC  |  SAU')
cv2_imshow(strip)

# Vẽ ĐÚNG vùng vừa bị nhuộm. Nhìn thẳng vào nó là biết ngay lỗi nằm ở mask hay ở màu:
# đỏ trùm sang áo/vai/nền -> mask bắt quá tay; đỏ hụt ở mảng tóc -> mask bắt thiếu.
if recolor is not None and recolor.last_alpha is not None:
    bx1, by1, bx2, by2 = recolor.last_box
    ov = before.copy()
    sub_ov = ov[by1:by2, bx1:bx2].astype(np.float32)
    a3 = np.clip(recolor.last_alpha, 0, 1)[..., None] * 0.6
    red = np.zeros_like(sub_ov); red[..., 2] = 255
    ov[by1:by2, bx1:bx2] = (sub_ov * (1 - a3) + red * a3).astype(np.uint8)
    ov = cv2.resize(ov, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA) if sc < 1.0 else ov
    print()
    print('VÙNG ĐANG BỊ NHUỘM (đỏ = nhuộm, đậm nhạt = mức độ):')
    cv2_imshow(ov)
    print('  đỏ trùm sang áo / vai / nền -> nâng HAIR_CONF, vd (0.5, 0.8)')
    print('  đỏ hụt mất mảng tóc          -> hạ HAIR_CONF, vd (0.25, 0.5)')

if recolor is not None:
    # ---- Đang nhuộm màu gì, và màu đó có ra được không? ----
    # Chỉ in màu đích là chưa đủ: câu hỏi thật sự là "tóc ĐEN trong video có với tới màu đích
    # không", mà chuyện đó do HAIR_LIGHTNESS quyết định chứ không phải do màu đích.
    print()
    print('--- Đang nhuộm màu gì ---')
    try:
        vid_bgr, cover = recolor.hair_color_of(before)
        Lv = float(to_lab(vid_bgr.reshape(1, 1, 3))[0])
        print(f'  tóc trong VIDEO : {bgr_to_hex(vid_bgr)}  L = {Lv:5.1f}/255'
              f'   (tóc chiếm {cover:.1%} khung hình)')
    except AssertionError:
        Lv = None
        print('  tóc trong VIDEO : không đo được (frame này không thấy tóc)')

    Lt = float(recolor.target_lab[0])
    nguon = ('màu tóc lấy từ ẢNH NGUỒN' if HAIR_COLOR == 'ảnh'
             else f'HAIR_COLOR = {HAIR_COLOR!r} - KHÔNG liên quan tới ảnh nguồn')
    print(f'  màu ĐÍCH        : {bgr_to_hex(recolor.target_bgr)}  L = {Lt:5.1f}/255   ({nguon})')

    if Lv is not None:
        L_pred = Lv + (Lt - Lv) * HAIR_LIGHTNESS
        print(f'  -> HAIR_LIGHTNESS = {HAIR_LIGHTNESS} nên độ sáng tóc sau khi nhuộm ~ L = {L_pred:.0f}/255')
        gap = Lt - Lv
        if abs(gap) > 40 and abs(L_pred - Lt) > 25:
            print()
            print(f'  !! Tóc trong video và màu đích lệch nhau {gap:+.0f} độ sáng, mà HAIR_LIGHTNESS')
            print(f'     chỉ {HAIR_LIGHTNESS} -> tóc dừng ở L={L_pred:.0f}, KHÔNG tới được L={Lt:.0f} của màu đích.')
            if gap > 0:
                print('     Tức là sẽ ra XÁM/nhạt chứ không ra đúng màu sáng bạn chọn.')
                print('     Sửa: HAIR_LIGHTNESS = 1.0  (thêm HAIR_CONTRAST = 1.3 cho đỡ bệt)')
            else:
                print('     Tức là tóc sẽ chưa đủ tối so với màu bạn chọn.')
                print('     Sửa: HAIR_LIGHTNESS = 1.0')

    print()
    print(f'Đã nhuộm: {"CÓ" if STATS["haired"] else "KHÔNG (không thấy tóc trong frame này)"}')
    print()
    print('Thấy gì thì chỉnh nấy (ở mục 0, rồi chạy lại mục 0 + mục 5b + cell này):')
    print('  màu không hiện ra trên tóc tối    -> tăng HAIR_LIGHTNESS (1.0)')
    print("  ra màu lạ, không giống ảnh nguồn  -> đặt HAIR_COLOR = 'ảnh'")
    print('  tóc sáng quá, trông dán vào cảnh  -> giảm HAIR_LIGHTNESS')
    print('  tóc bệt, phẳng, mất sợi           -> tăng HAIR_CONTRAST (1.2-1.4)')
    print('  màu trông như sơn, quá đều        -> tăng HAIR_KEEP_TONE (0.4-0.6)')
    print('  tóc bạc trắng ở chỗ bắt sáng      -> giảm HAIR_KEEP_HIGHLIGHT')
    print('  tóc mất bóng, nhìn như tóc giả    -> tăng HAIR_KEEP_HIGHLIGHT')
    print('  QUẦNG SÁNG viền quanh tóc, hoặc')
    print('  viền TỐI còn màu tóc cũ ở mép     -> kiểm tra HAIR_EDGE_SMART = True')
    print('  sợi tóc mai trước mặt vẫn màu cũ  -> tăng HAIR_WISPS (0.8-1.0)')
    print('  nhuộm nhầm vệt tối trên mặt/cổ    -> giảm HAIR_WISPS (0.4 hoặc 0)')
    print('  màu lem sang nền / da / vai       -> nâng HAIR_CONF, vd (0.5, 0.8)')
    print('  rìa tóc chưa ăn màu               -> hạ HAIR_CONF, vd (0.25, 0.5)')
    print('  màu quá rực / quá nhạt            -> chỉnh HAIR_SATURATION')
    print('  muốn nhẹ tay, giữ chút màu gốc    -> giảm HAIR_STRENGTH (0.6-0.8)')

In [ ]:
# Thử NHIỀU màu cùng lúc trên đúng một frame, để chọn bằng mắt thay vì đoán.
# Mỗi màu chỉ tốn thêm một lượt nhuộm (~vài chục ms), không phải xử lý lại cả video.
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

TRY_COLORS = ['nâu hạt dẻ', 'nâu tây', 'vàng đồng', 'đỏ rượu', 'bạch kim', 'tím khói']
TRY_AT     = 0.5      # vị trí frame trong video, giống PREVIEW_AT

if recolor is None:
    print('USE_HAIR = False (hoặc mediapipe không dùng được) -> không có gì để thử.')
else:
    cap = cv2.VideoCapture(source_video_path)
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n_total > 0:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(n_total * TRY_AT))
    ok, base = cap.read()
    if not ok:
        cap.release(); cap = cv2.VideoCapture(source_video_path); ok, base = cap.read()
    cap.release()
    assert ok, 'Không đọc được frame nào từ video.'

    # Swap mặt 1 lần rồi dùng lại cho mọi màu: phần swap không phụ thuộc màu tóc, chạy lại
    # từng lần chỉ tốn thời gian mà cho ra đúng kết quả cũ.
    face = pick_face(app.get(base))
    swapped = base if face is None else swapper.get(base, face, source_face, paste_back=True)
    if face is not None and restorer is not None:
        swapped = enhance_face_region(swapped, face)

    saved_target = recolor.target_bgr.copy()
    tiles, labels = [swapped.copy()], ['(gốc)']
    for nm in TRY_COLORS:
        hx = HAIR_PALETTE.get(nm, nm)
        recolor.set_target(hex_to_bgr(hx)[0, 0])
        recolor.reset()
        kps = None if face is None else face.kps
        img = swapped.copy()
        recolor.apply(img, kps)    # lần 1: nạp EMA
        img = swapped.copy()
        recolor.apply(img, kps)    # lần 2: đúng trạng thái như khi chạy video
        tiles.append(img)
        labels.append(nm)
    recolor.set_target(saved_target)   # trả lại màu đã chọn ở mục 0
    recolor.reset()

    # Cắt quanh đầu cho dễ so sánh, rồi xếp lưới.
    if face is not None:
        x1, y1, x2, y2 = face.bbox.astype(int)
        bw, bh = x2 - x1, y2 - y1
        cx1 = max(0, int(x1 - 1.1 * bw)); cx2 = min(base.shape[1], int(x2 + 1.1 * bw))
        cy1 = max(0, int(y1 - 1.4 * bh)); cy2 = min(base.shape[0], int(y2 + 1.2 * bh))
        tiles = [t[cy1:cy2, cx1:cx2] for t in tiles]

    tw = 260
    tiles = [cv2.resize(t, (tw, int(t.shape[0] * tw / t.shape[1])),
                        interpolation=cv2.INTER_AREA) for t in tiles]
    th = min(t.shape[0] for t in tiles)
    tiles = [t[:th] for t in tiles]

    cols = 4
    rows = [np.hstack(tiles[i:i + cols] + [np.zeros((th, tw, 3), np.uint8)]
                      * ((cols - len(tiles[i:i + cols])) % cols))
            for i in range(0, len(tiles), cols)]
    cv2_imshow(np.vstack(rows))
    print('Thứ tự trái->phải, trên->xuống:')
    for i, nm in enumerate(labels):
        print(f'  {i}. {nm}')
    print()
    print('Chọn được rồi thì đặt HAIR_COLOR ở mục 0, chạy lại mục 0 + mục 5b, rồi chạy mục 6.2.')

In [ ]:
import os, subprocess, time
import numpy as np
import cv2
from tqdm import tqdm

cap = cv2.VideoCapture(source_video_path)
assert cap.isOpened(), f'Không mở được video: {source_video_path}'

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps != fps or fps <= 0:       # 0.0 hoặc NaN với một số container
    print('Không đọc được fps từ video, mặc định 25.')
    fps = 25.0

# CAP_PROP_FRAME_COUNT chỉ là ước lượng (sai với VFR / mp4 thiếu index) -> chỉ dùng cho progress bar.
# Vòng lặp đọc tới khi hết frame thật sự, thay vì range(total_frames) (đếm thiếu = cụt đuôi video).
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or None

# Lấy kích thước từ frame THẬT, không từ CAP_PROP_FRAME_WIDTH/HEIGHT: video quay dọc có metadata
# rotation làm hai giá trị đó bị hoán đổi so với frame OpenCV trả về -> ffmpeg nhận rawvideo sai size.
ret, frame = cap.read()
assert ret, 'Không đọc được frame nào từ video.'
height, width = frame.shape[:2]

print(f'Video: {width}x{height} @ {fps:.2f}fps, ~{total_frames} frames')

# Ghi frame thô thẳng vào ffmpeg và ghép audio ngay trong cùng một pass.
ffmpeg_cmd = [
    'ffmpeg', '-y', '-loglevel', 'error',
    '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{width}x{height}', '-r', f'{fps}', '-i', 'pipe:0',
    '-i', source_video_path,
    '-map', '0:v:0', '-map', '1:a:0?',        # '?' = không có audio thì bỏ qua, không lỗi
    '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
    '-pix_fmt', 'yuv420p',                    # để trình duyệt/IPython.display.Video phát được
    '-c:a', 'aac', '-shortest',
    final_output,
]
proc = subprocess.Popen(ffmpeg_cmd, stdin=subprocess.PIPE)

reset_state()                                  # xoá số liệu + trạng thái EMA của cell preview
pbar = tqdm(total=total_frames, unit='frame')
t_start = time.perf_counter()
rc = None
try:
    while frame is not None:
        result_frame = process_frame(frame)

        proc.stdin.write(np.ascontiguousarray(result_frame).tobytes())
        pbar.update(1)

        ret, frame = cap.read()
        if not ret:
            frame = None
finally:
    # Không release trong finally thì khi bấm Stop giữa chừng, ffmpeg treo và file mp4 hỏng.
    pbar.close()
    cap.release()
    try:
        proc.stdin.close()
    except BrokenPipeError:
        pass
    rc = proc.wait()

wall = time.perf_counter() - t_start
assert rc == 0, f'ffmpeg thất bại (exit code {rc}) - xem log lỗi ở trên.'

n  = max(STATS['frames'], 1)
ns = max(STATS['swapped'], 1)

print()
print(f'Xong: {STATS["frames"]} frame, trong đó {STATS["swapped"]} frame có mặt để swap '
      f'({STATS["swapped"] / n:.0%}).')
if recolor is not None:
    print(f'      nhuộm tóc {STATS["haired"]}/{STATS["frames"]} frame ({STATS["haired"] / n:.0%})'
          f' - tính trên MỌI frame, không chỉ frame thấy mặt.')
    if STATS['haired'] > STATS['swapped']:
        print(f'      (trong đó {STATS["haired"] - STATS["swapped"]} frame nhuộm được tóc dù '
              f'không detect ra mặt - quay lưng, cúi đầu, mặt bị che)')
    elif STATS['haired'] < n * 0.9:
        print('      tỉ lệ nhuộm thấp -> thử hạ HAIR_CONF, hoặc video có đoạn không thấy tóc.')
print(f'Output: {final_output}')

# ---- Thời gian từng bước: để biết cái nào tốn, thay vì đoán ----
print()
print('Thời gian trung bình mỗi frame:')
print(f'  detect mặt : {STATS["detect"] / n * 1000:7.1f} ms')
print(f'  swap       : {STATS["swap"] / ns * 1000:7.1f} ms  (tính trên frame có mặt)')
print(f'  làm nét    : {STATS["enhance"] / ns * 1000:7.1f} ms'
      f'{"" if restorer is not None else "      (TẮT)"}')
print(f'  nhuộm tóc  : {STATS["hair"] / n * 1000:7.1f} ms  (tính trên MỌI frame)'
      f'{"" if recolor is not None else "   (TẮT)"}')

busy = STATS['detect'] + STATS['swap'] + STATS['enhance'] + STATS['hair']
if restorer is not None:
    print(f'-> làm nét chiếm {STATS["enhance"] / max(busy, 1e-9):.0%} thời gian xử lý '
          f'(USE_GFPGAN = False để bỏ).')
if recolor is not None:
    print(f'-> nhuộm tóc chiếm {STATS["hair"] / max(busy, 1e-9):.0%} thời gian xử lý.')

print()
print(f'Tổng: {wall:.1f}s cho {STATS["frames"]} frame ({wall / n * 1000:.0f} ms/frame, '
      f'{n / max(wall, 1e-9):.1f} fps xử lý).')

## 7. Kiểm tra kết quả

Audio đã được ghép ngay trong cell trên (ffmpeg nhận frame qua pipe và mux luôn audio gốc
trong cùng một pass), nên ở đây chỉ cần xác nhận file xuất ra hợp lệ.

In [ ]:
import os

assert os.path.exists(final_output) and os.path.getsize(final_output) > 0, \
    'Không tạo được video output — chạy lại cell xử lý video ở mục 6.'
print(f'{final_output}  —  {os.path.getsize(final_output) / 1e6:.1f} MB')
print()

# Xác nhận có stream video (và audio, nếu video gốc có audio)
!ffprobe -v error -show_entries stream=index,codec_type,codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 {final_output}

## 8. Xem kết quả

In [ ]:
import os
from IPython.display import Video, display

size_mb = os.path.getsize(final_output) / 1e6
if size_mb > 50:
    # embed=True nhét toàn bộ file dưới dạng base64 vào output của notebook -> file .ipynb phình to
    # và trình duyệt dễ treo với video dài. Trường hợp đó thì tải về xem thay vì preview inline.
    print(f'Video {size_mb:.1f} MB — quá lớn để nhúng inline, chạy cell dưới để tải về máy.')
else:
    display(Video(final_output, embed=True, width=480))

In [ ]:
# Tải file về máy
from google.colab import files
files.download(final_output)

## Ghi chú / Giới hạn / Khi nào dùng notebook nào

### Cách chỉnh cho nhanh

Đừng chạy cả video để thử. Vòng đúng là: **sửa mục 0 → chạy lại cell mục 0 → chạy lại cell mục
5b** (để nạp màu mới vào `recolor`) **→ chạy lại cell 6.1**. Vài giây một vòng. Muốn so nhiều
màu thì dùng cell **6.1b** — nó nhuộm 6 màu lên cùng một frame và xếp thành lưới.

Thử thêm vài mốc `PREVIEW_AT` khác nhau (0.1 / 0.5 / 0.9) để bắt cả những đoạn ánh sáng khác.

### Chọn tham số theo tình huống

| Muốn gì | Chỉnh |
|---|---|
| **Tóc giống hệt người trong ảnh nguồn** | **`HAIR_COLOR = 'ảnh'`** — mặc định là tên màu cố định, không lấy từ ảnh |
| Đen → nâu/hạt dẻ (dễ nhất, tự nhiên nhất) | mặc định là đủ |
| **Đen → trắng / bạch kim / vàng** (khó) | `HAIR_COLOR` + **`HAIR_LIGHTNESS = 1.0`** và `HAIR_CONTRAST` 1.2-1.4 |
| Quầng sáng quanh tóc, hoặc viền tối ở mép | `HAIR_EDGE_SMART = True` (mặc định) |
| Sợi tóc mai trước mặt vẫn màu cũ | tăng `HAIR_WISPS` (mặc định 0.7) |
| Nhuộm lây vào vệt tối trên mặt/cổ | giảm `HAIR_WISPS` |
| Chỉ đổi sắc, giữ nguyên độ sáng | `HAIR_LIGHTNESS = 0` |
| Nhuộm nhẹ, còn thấy màu gốc | `HAIR_STRENGTH` 0.6-0.8 |
| Màu hiện đại, phẳng, "ăn ảnh" | `HAIR_KEEP_TONE` 0-0.15 |
| Tự nhiên như tóc thật | `HAIR_KEEP_TONE` 0.4-0.6, `HAIR_KEEP_HIGHLIGHT` 0.7-0.85 |

### Giới hạn đã biết (không phải bug)

- **Màu ra không giống màu mình tưởng?** Hai nguyên nhân, theo thứ tự hay gặp: (1) `HAIR_COLOR`
  vẫn là tên màu mặc định chứ chưa đặt `'ảnh'`; (2) `HAIR_LIGHTNESS` quá thấp so với khoảng cách
  độ sáng giữa tóc trong video và màu đích. Cell 6.1 in ra cả ba con số (tóc video / màu đích /
  kết quả dự kiến) nên nhìn là biết ngay cái nào.
- **Vẫn thấy mảng màu tóc cũ ở đỉnh đầu / chỗ rẽ ngôi?** Đó là chỗ tóc bắt sáng.
  `HAIR_KEEP_HIGHLIGHT` cố ý chừa bớt màu ở đó cho tóc còn độ bóng — để cao quá thì thành ra
  nhìn như sót màu cũ. Hạ về `0.2`, hoặc `0` nếu muốn nhuộm phẳng hoàn toàn.
- **Màu trùm sang áo / vai / nền?** Đó là mask của MediaPipe bắt quá tay, không phải phần đổi
  màu sai. Cell 6.1 vẽ sẵn **vùng đang bị nhuộm** (tô đỏ) — nhìn ảnh đó là biết ngay. Chữa bằng
  cách nâng `HAIR_CONF`, ví dụ `(0.5, 0.8)` hoặc `(0.6, 0.85)`. Hay gặp khi tóc tối đổ bóng lên
  áo sáng màu, hoặc nền tối gần màu tóc.
- **Sợi tóc mai trước mặt vẫn đen?** `HAIR_WISPS` lo phần này (mặc định `0.7`). Sợi rộng 4-8
  pixel là chưa tới một ô của mask 256x256 nên MediaPipe không thấy, phải bắt riêng bằng độ
  tương phản sáng/tối. Mắt/mũi/miệng/lông mày được chừa sẵn. Ngược lại, nếu thấy nó nhuộm nhầm
  vào vệt tối trên mặt/cổ/viền áo thì giảm xuống `0.4` hoặc `0`.
  Nó **chỉ ăn những sợi nằm gần mái tóc** (trong khoảng 2 lần khoảng cách hai mắt) và sáng/tối
  rõ rệt so với da — sợi quá mảnh, quá mờ hoặc trùng màu da thì vẫn chịu.
- **Mép tóc là chỗ khó nhất, và càng lệch độ sáng càng khó.** Pixel ở đúng biên tóc/nền là màu
  PHA giữa hai thứ, mà mask 256×256 thì quá thô để biết tỉ lệ pha. Chỉ chỉnh độ rộng mask thì
  chỉ đổi lỗi này lấy lỗi kia: rộng ra thành **quầng sáng** ở nền, hẹp lại thành **viền tối**
  còn màu tóc cũ. `HAIR_EDGE_SMART` thoát khỏi thế đó bằng cách đọc tỉ lệ pha từ chính độ sáng
  của ảnh. Nó **chỉ bất lực khi tóc và nền sáng xấp xỉ nhau** (tóc đen trên nền tối) — lúc đó
  không có thông tin nào để tách, và nó tự quay về dùng mask như cũ.
- **Tóc gần đen nhuộm sang màu sáng sẽ phẳng hơn tóc vốn đã sáng.** Pixel tối gần như không còn
  biến thiên độ sáng để giữ — ngoài đời cũng phải tẩy tóc trước mới nhuộm sáng được.
  `HAIR_CONTRAST` bù lại được một phần, nhưng không tạo ra chi tiết vốn không có trong ảnh.
- **MediaPipe nhuộm mọi mái tóc nó thấy.** Video có người khác trong khung thì tóc họ cũng đổi
  màu. Model `selfie_multiclass` vốn để cho ảnh selfie một người nên thường bám chủ thể chính,
  nhưng không có gì bảo đảm. Đây là notebook 1 người.
- **Mask sai ở chỗ tóc trùng màu nền** (tóc đen trên nền tối), khi đội mũ, hoặc tóc bị tay che.
  Xem trực tiếp mask ở cell preview mục 5b trước khi chạy cả video.
- **Tóc mảnh, bay lơ thơ ở rìa** không được nhuộm sạch: mask ở 256×256 không đủ mịn cho từng
  sợi. Hạ `HAIR_CONF` giúp được một phần, đổi lại dễ lem sang nền.
- **Vùng bắt sáng mạnh** (đèn chiếu thẳng) giữ gần như nguyên màu gốc — đó là chủ ý
  (`HAIR_KEEP_HIGHLIGHT`), giảm xuống nếu bạn muốn nhuộm cả vào đó.

### Notebook nào cho việc nào

| Muốn | Dùng |
|---|---|
| Chỉ thay mặt | `video_face_swap_1nguoi.ipynb` |
| Thay mặt + **đổi màu tóc** | **notebook này** |
| Thay mặt + **thay hẳn kiểu tóc** từ ảnh nguồn | `video_face_swap_1nguoi_toc.ipynb` |
| Thay mặt cho **hai người** (cảnh hôn) | `video_face_swap_kiss.ipynb` |

Đổi màu tóc **ổn định hơn hẳn** thay kiểu tóc: không warp, không phụ thuộc góc quay đầu, không
inpaint, không hộp ROI. Nếu đổi màu là đủ cho việc bạn cần thì nên dùng bản này.

### Phần còn lại (giữ từ bản gốc)

- **Bật/tắt làm nét / nhuộm tóc**: đổi ở **mục 0** rồi chạy lại từ đầu. Đổi giữa chừng từ `False`
  sang `True` thì phải chạy lại mục 1 (cài package), mục 2 (tải weights) và mục 5/5b.
- **Video nhiều người mà muốn thay người khác, không phải người to nhất**: sửa `pick_face()` ở
  mục 6.0. Cần thay **hai người khác nhau** thì dùng `video_face_swap_kiss.ipynb`.
- **Flicker giữa các frame**: face swap từng frame độc lập nên đôi khi có giật/nhoè. GFPGAN tự
  nó cũng gây flicker vì "sáng tác" chi tiết khác nhau mỗi frame. Phần tóc đã có EMA
  (`HAIR_MASK_SMOOTH`) nên ổn định hơn phần mặt.
- **`inswapper_128`**: link tải có thể thay đổi do vấn đề chính sách/gỡ bỏ. Cell mục 2 tự kiểm
  tra dung lượng file và báo lỗi ngay nếu tải hỏng.